# Reinforcement Learning — PPO Training

This notebook **fine-tunes the BC model using PPO** (Proximal Policy Optimization).

**Prerequisites:**
- Run the BC training notebook first (produces `model.pt`)
- Add the BC notebook's output as a data source in this notebook

**What this notebook does:**
1. Loads the BC model from the data source
2. Defines the RL environment, actor-critic, and PPO training loop
3. Warm-starts the RL policy from BC weights
4. Trains via PPO against the opponent pool
5. Saves `model_rl.pt` for submission


## Setup & Imports


In [ ]:
import sys
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Engine path (Kaggle competition data)
CG_PATH = "/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/"
sys.path.append(CG_PATH)

from cg.game import battle_start, battle_finish, battle_select
from cg.api import (AreaType, CardType, Log, LogType, Observation, SelectContext,
                    OptionType, Card, Pokemon, State, all_card_data, to_observation_class)
from collections import defaultdict

print("Imports OK")

## Deck


In [ ]:
# --- Pokemon ---
# Decklist
Dreepy = 119  # ×4
Drakloak = 120  # ×4
Dragapult_ex = 121  # ×3
Fezandipiti_ex = 140  # ×1
Latias_ex = 184  # ×1
Budew = 235  # ×2
Meowth_ex = 1071  # ×1
Rare_Candy = 1079  # ×2
Unfair_Stamp = 1080  # ×1
Buddy_Buddy_Poffin = 1086  # ×4
Night_Stretcher = 1097  # ×2
Crushing_Hammer = 1120  # ×4
Ultra_Ball = 1121  # ×4
Poke_Pad = 1152  # x3
Lucky_Helmet = 1156  # ×1
Boss_Orders = 1182  # ×3
Crispin = 1198  # ×4
Brock_Scouting = 1210  # ×2
Lillie_Determination = 1227  # ×4
Team_Rocket_Watchtower = 1256  # ×2
Basic_Fire_Energy = 2  # ×4
Basic_Psychic_Energy = 5  # ×4

# ==========================================================
# DECK (must total exactly 60 cards)
DRAGAPULT_DECK_RECIPE = [
    # Pokemon (16)
    (Dreepy, 4), (Drakloak, 4), (Dragapult_ex, 3),
    (Fezandipiti_ex, 1), (Latias_ex, 1), (Budew, 2), (Meowth_ex, 1),
    # Trainers - Items
    (Rare_Candy, 2), (Unfair_Stamp, 1), (Buddy_Buddy_Poffin, 4),
    (Night_Stretcher, 2), (Crushing_Hammer, 4), (Ultra_Ball, 4),
    (Poke_Pad, 3), (Lucky_Helmet, 1),
    # Trainers - Supporters
    (Boss_Orders, 3), (Crispin, 4), (Brock_Scouting, 2),
    (Lillie_Determination, 4),
    # Stadium
    (Team_Rocket_Watchtower, 2),
    # Energy (8)
    (Basic_Fire_Energy, 4), (Basic_Psychic_Energy, 4),
]
def build_deck(recipe):
    deck = []
    for card_id, count in recipe:
        deck.extend([card_id] * count)
    return deck

DRAGAPULT_DECK = build_deck(DRAGAPULT_DECK_RECIPE)
assert len(DRAGAPULT_DECK) == 60, f"Deck must be 60 cards, got {len(DRAGAPULT_DECK)}"

## Expert Heuristic Agent (for opponent pool)


In [ ]:
# Load all card data from the API's helper function
all_card = all_card_data()
# Create a lookup table (dictionary) to quickly access card data by its cardId
card_table = {c.cardId:c for c in all_card}

UNNECESSARY = -10000000

class AttackPlan:
    attack: int = 0
    counter: list[int] = []

can_switch = False
can_attack = False
can_main_attack = False
can_energy_attach = False
use_support = 0  # The Supporter card planned for use.
bench_attacker = False  # Whether there is a Benched Pokémon that is ready to attack
pre_turn_log: list[Log] = []
current_turn_log: list[Log] = []

prize: list[int] = []
card_counts: defaultdict[int, int] = defaultdict(int)
serial_set: set[int] = set()
plan_a = AttackPlan()
plan_b = AttackPlan()


def no_damage_dex(id: int) -> bool:
    """Checks if the defending Pokémon possesses innate immunities preventing Dragapult ex from hitting it."""
    # Drednaw, Milotic ex, Sylveon, Crustle
    return id == 158 or id == 207 or id == 330 or id == 345


def no_damage_counter(pokemon: Pokemon) -> bool:
    """Checks if a target prevents placement of Phantom Dive's 6 bench damage counters (via abilities/Energy)."""
    # Poltchageist, Empoleon ex, Skeledirge, Milotic ex, Misty's Magikarp, Antique Cover Fossil
    if pokemon.id == 28 or pokemon.id == 199 or pokemon.id == 203 or pokemon.id == 207 or pokemon.id == 362 or pokemon.id == 1136:
        return True
    for card in pokemon.energyCards:
        # Mist Energy, Rock Fighting Energy
        if card.id == 11 or card.id == 20:
            return True
    return False


def prize_count(pokemon: Pokemon, is_attack_damage: bool) -> int:
    """Calculates how many Prize cards a Pokémon yields upon being Knocked Out, factoring in modifiers."""
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    if is_attack_damage:
        for card in pokemon.energyCards:
            if card.id == 12:  # Legacy Energy
                count -= 1
        for card in pokemon.tools:
            if card.id == 1172 and "Lillie" in data.name:  # Lillie’s Pearl
                count -= 1
    return max(0, count)


def pokemon_score(pokemon: Pokemon, is_attack_damage: bool) -> int:
    """Heuristically evaluates the tactical worth of targeting a specific Pokémon on the opponent's field."""
    data = card_table[pokemon.id]
    score = prize_count(pokemon, is_attack_damage) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130
    
    id = pokemon.id
    # Noctowl, Fan Rotom, Archaludon ex, Meowth ex
    if id == 173 or id == 174 or id == 190 or id == 1071:
        score -= 200
    if id == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score


def add_card_count(card: Card | Pokemon | None, my_index: int):
    if card == None:
        return
    if isinstance(card, Pokemon) or card.playerIndex == my_index:
        if card.serial not in serial_set:
            card_counts[card.id] -= 1
            serial_set.add(card.serial)
    if isinstance(card, Pokemon):
        for c in card.energyCards:
            add_card_count(c, my_index)
        for c in card.tools:
            add_card_count(c, my_index)
        for c in card.preEvolution:
            add_card_count(c, my_index)

def set_card_counts(obs: Observation, my_index: int):
    card_counts.clear()
    serial_set.clear()
    for id in DRAGAPULT_DECK:
        card_counts[id] += 1
    
    state = obs.current
    my_state = state.players[my_index]
    for card in my_state.hand:
        add_card_count(card, my_index)
    for card in my_state.discard:
        add_card_count(card, my_index)
    for card in my_state.bench:
        add_card_count(card, my_index)
    for card in my_state.active:
        add_card_count(card, my_index)
    for card in state.stadium:
        add_card_count(card, my_index)
    if state.looking != None:
        for card in state.looking:
            add_card_count(card, my_index)
    add_card_count(obs.select.effect, my_index)

    
def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Helper function to safely extract a Card or Pokemon object from specific zones."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def main_option_proc(obs: Observation, damage: int):
    state = obs.current
    select = obs.select
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]

    global can_switch
    global can_attack
    global can_main_attack
    global can_energy_attach

    can_switch = False
    can_attack = False
    can_main_attack = False
    can_energy_attach = False
    for o in select.option:
        if o.type == OptionType.RETREAT:
            can_switch = True
        elif o.type == OptionType.ATTACK:
            can_attack = True
            if o.attackId == 154:  # Phantom Dive
                can_main_attack = True
    
    plan_a.attack = -1
    plan_b.attack = -1
    if not can_main_attack and not (bench_attacker and can_switch):
        return
    
    cards = [op_state.active[0]]
    for pokemon in op_state.bench:
        cards.append(pokemon)
    counter_indices = []
    ci = []
    ci.append(0)
    remain_damage = 60
    while ci:
        index = ci[-1]
        hp = cards[index].hp
        if remain_damage >= hp:
            counter_indices.append(ci.copy())
            if index < len(cards) - 1:
                remain_damage -= hp
                ci.append(index + 1)
                continue
        if index == len(cards) - 1:
            ci.pop()
            if ci:
                remain_damage += cards[ci[-1]].hp
        if ci:
            ci[-1] += 1
    counter_indices.append([])

    remain_prize = len(my_state.prize)
    plan_score = 0
    for i, pokemon in enumerate(cards):
        base_prize_count = 0
        base_score = pokemon_score(pokemon, True)
        active_damage = 0 if no_damage_dex(pokemon.id) else damage
        if pokemon.hp <= active_damage:
            base_prize_count += prize_count(pokemon, True)
        else:
            base_score *= active_damage / pokemon.hp
        ci = []
        max_score = base_score
        if remain_prize <= base_prize_count:
            max_score = 50000
        else:
            for indices in counter_indices:
                if i in indices:
                    continue
                prize = base_prize_count
                score = base_score
                for index in indices:
                    prize += prize_count(cards[index], False)
                    score += pokemon_score(cards[index], False)
                if remain_prize <= prize:
                    score = 50000
                else:
                    if prize >= 2:
                        if remain_prize <= 4:
                            score -= 1200
                    elif prize == 1:
                        score -= 300
                    else:
                        score += 1200
                if max_score < score:
                    max_score = score
                    ci = indices
        if plan_score < max_score:
            plan_score = max_score
            plan_a.attack = i
            plan_a.counter = ci
        if i == 0:
            plan_b.attack = plan_a.attack
            plan_b.counter = plan_a.counter

In [ ]:
def EXPERT_DRAGAPULT_AGENT(obs_dict: dict) -> list[int]:
    """Main Agent Function.

    Each element in the returned list must be >= 0 and < len(obs.select.option).
    The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.
    
    Returns:
        list[int]: A list of option index.
    """
    obs = to_observation_class(obs_dict)
    if obs.select == None:
        # In the initial selection, the obs.select is None, and it is necessary to return the deck.
        # The deck is a list of 60 card IDs.
        # The deck must comply with the Pokémon Trading Card Game rules.
        return DRAGAPULT_DECK

    global pre_turn_log
    global current_turn_log

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
            
    if state.turn == 0:
        prize.clear()
        pre_turn_log.clear()
        current_turn_log.clear()
    else:
        for log in obs.logs:
            current_turn_log.append(log)
            if log.type == LogType.TURN_END:
                pre_turn_log = current_turn_log
                current_turn_log = []

    pre_ko = False
    no_item = False
    for log in pre_turn_log:
        if log.type == LogType.ATTACK:
            if log.attackId == 323:  # Itchy Pollen
                no_item = True
        elif log.type == LogType.MOVE_CARD:
            if (log.playerIndex == my_index
                and (log.fromArea == AreaType.BENCH or log.fromArea == AreaType.ACTIVE)
                and log.toArea == AreaType.DISCARD):
                pre_ko = True

    if select.deck != None:
        set_card_counts(obs, my_index)
        for card in select.deck:
            card_counts[card.id] -= 1
        prize.clear()
        for id in card_counts:
            for _ in range(card_counts[id]):
                prize.append(id)
                
    set_card_counts(obs, my_index)
    for id in prize:
        card_counts[id] -= 1
    deck_counts = card_counts

    prize_diff = len(my_state.prize) - len(op_state.prize)
    
    global bench_attacker

    # Number of cards per card ID on the Bench and in the Active Spot
    field_counts = defaultdict(int)
    # Number of cards per card ID in hand
    hand_counts = defaultdict(int)
    # Number of cards per card ID in discard pile
    discard_counts = defaultdict(int)
    
    active_id = 0
    bench_attacker = False
    can_evolve_dreepy = False
    evolve_dreepy_count = 0
    can_evolve_drakloak = False
    damage = 200
    for card in my_state.active:
        if card == None:
            continue
        active_id = card.id
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
    for card in my_state.bench:
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
        if card.id == Dragapult_ex and len(card.energies) >= 2:
            bench_attacker = True
    main_pokemon_count = field_counts[Dreepy] + field_counts[Drakloak] + field_counts[Dragapult_ex]
    no_more_dex = (field_counts[Dragapult_ex] * 2 >= len(op_state.prize))

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    support_count = 0

    for card in my_state.discard:
        discard_counts[card.id] += 1

    def attach_score(attach_id: int, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        if card_table[attach_id].cardType == CardType.TOOL:
            # Attach tool
            score = 60000
            if active:
                score += 1000
            return score
        
        # Attach energy
        if pokemon.id == Budew:
            return -1
        elif pokemon.id == Meowth_ex or pokemon.id == Fezandipiti_ex or pokemon.id == Latias_ex:
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                if bench_attacker or field_counts[Budew] >= 1:
                    return 22000
                else:
                    return 18000
            else:
                return -1
        if active and can_main_attack:
            return -1
        score = 20000
        if energy_count >= 2:
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                score += 200
            else:
                return -1
        elif energy_count == 1:
            if attach_id == pokemon.energyCards[0].id:
                return -1
            if pokemon.id == Dragapult_ex:
                score += 250
            elif pokemon.id == Dreepy:
                score -= 150
            else:
                score -= 200
            if active:
                score += 200
        else:  # energy_count == 0
            if active:
                if bench_attacker:
                    score += 400
            else:
                if pokemon.id == Dragapult_ex:
                    score += 150
                elif pokemon.id == Dreepy:
                    score += 100
                else:
                    score += 50
                if bench_attacker:
                    score -= 200
        if no_more_dex and (pokemon.id == Dreepy or pokemon.id == Drakloak):
            score -= 500
        return score
    
    def hand_score(id: int, ignore_count: bool):
        score = 0
        if id == Dreepy:
            if main_pokemon_count >= 3:
                score = 1000
            else:
                score = 18000
        elif id == Drakloak:
            if can_evolve_dreepy:
                score = 20000
            else:
                score = 3000
        elif id == Dragapult_ex:
            if no_more_dex:
                score = UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Rare_Candy] >= 1 and not no_item:
                score = 40000
            elif can_evolve_drakloak:
                if field_counts[id] == 0:
                    score = 30000
                elif field_counts[id] == 1:
                    score = 10000
                else:
                    score = 50
            else:
                if field_counts[id] >= 2:
                    score = 50
                else:
                    score = 2000
        elif id == Fezandipiti_ex:
            if pre_ko:
                score = 50000
            elif prize_diff <= -2:
                score = 5
            elif len(op_state.prize) == 1:
                score = UNNECESSARY
        elif id == Latias_ex:
            if active_id == Fezandipiti_ex or active_id == Meowth_ex or active_id == Dreepy:
                if field_counts[Drakloak] + field_counts[Dragapult_ex] == 0:
                    score = 28000
                else:
                    score = 15000
            else:
                score = 10
        elif id == Budew:
            if field_counts[id] + field_counts[Drakloak] + field_counts[Dragapult_ex] >= 1:
                score = UNNECESSARY
            elif state.turn >= 2:
                score = 30000
        elif id == Meowth_ex:
            if support_count > hand_counts[Boss_Orders] or stadium_id == Team_Rocket_Watchtower:
                score = 5
            elif state.supporterPlayed:
                score = 40
            else:
                score = 35000
        elif id == Rare_Candy:
            if no_more_dex:
                score = UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Dragapult_ex] >= 1:
                score = 40000
        elif id == Unfair_Stamp:
            if pre_ko:
                score = 80000
            elif len(op_state.prize) == 1:
                score = UNNECESSARY
            else:
                score = 80
        elif id == Buddy_Buddy_Poffin:
            count = deck_counts[Dreepy]
            if count == 0:
                score = UNNECESSARY
            else:
                if state.turn <= 2 and field_counts[Budew] == 0 and deck_counts[Budew] >= 1:
                    count += 1
                if count >= 2:
                    score = 35000
        elif id == Night_Stretcher:
            for i in discard_counts:
                if discard_counts[i] >= 1:
                    card_type = card_table[i].cardType
                    if card_type == CardType.POKEMON or card_type == CardType.BASIC_ENERGY:
                        score = max(score, hand_score(i, ignore_count))
        elif id == Crushing_Hammer:
            score = 20
        elif id == Ultra_Ball:
            if main_pokemon_count <= 2 or field_counts[Dreepy] >= 1:
                score = 70
            else:
                score = 5
        elif id == Poke_Pad:
            score = max(hand_score(Dreepy, ignore_count), hand_score(Drakloak, ignore_count))
        elif id == Lucky_Helmet:
            score = 15
        elif id == Boss_Orders:
            if plan_a.attack > 0:
                score = 60000
        elif id == Crispin:
            if not ignore_count or support_count == 0:
                if deck_counts[Basic_Fire_Energy] == 0 or deck_counts[Basic_Psychic_Energy] == 0:
                    score = 10
                if not can_main_attack and not bench_attacker and field_counts[Dragapult_ex] >= 1:
                    score = 55000
                else:
                    score = 25000
        elif id == Brock_Scouting:
            if not ignore_count or support_count == 0:
                if state.turn == 2 and field_counts[Budew] + field_counts[Latias_ex] == 0:
                    score = 50000
                else:
                    score = 30000
        elif id == Lillie_Determination:
            if not ignore_count or support_count == 0:
                score = 45000
        elif id == Team_Rocket_Watchtower:
            if stadium_id != 0 and stadium_id != Team_Rocket_Watchtower:
                score = 4000
        elif id == Basic_Fire_Energy or id == Basic_Psychic_Energy:
            if can_main_attack and (len(op_state.prize) <= 2
                or (bench_attacker and len(op_state.prize) <= 4)):
                score = UNNECESSARY
            else:
                max_score = -10000
                for pokemon in my_state.active:
                    if pokemon == None:
                        continue
                    max_score = max(max_score, attach_score(id, pokemon, True))
                for pokemon in my_state.bench:
                    max_score = max(max_score, attach_score(id, pokemon, False))
                score = max_score - 5000
                if can_main_attack or bench_attacker:
                    score /= 10
        
        if not ignore_count and hand_counts[id] > 0:
            if id == Drakloak and hand_counts[id] < evolve_dreepy_count:
                score -= 10
            elif id == Dreepy:
                score -= 100
            else:
                score -= 100000
        return score

    global use_support
    if context == SelectContext.MAIN:
        main_option_proc(obs, damage)
                    
        use_support = 0
        if not state.supporterPlayed:
            support_score = 0
            for o in select.option:
                if o.type == OptionType.PLAY:
                    card = get_card(obs, AreaType.HAND, o.index, state.yourIndex)
                    if card_table[card.id].cardType == CardType.SUPPORTER:
                        score = hand_score(card.id, True)
                        if support_score < score:
                            support_score = score
                            use_support = card.id

    hand_scores = []
    negative_hand_count = 0
    for card in my_state.hand:
        score = hand_score(card.id, False)
        hand_scores.append(score)
        if score < 0:
            negative_hand_count += 1
        hand_counts[card.id] += 1
        if card_table[card.id].cardType == CardType.SUPPORTER and card.id != Boss_Orders:
            support_count += 1

    no_draw = (my_state.deckCount <= 8)  # Whether to restrict actions that reduce the deck
    do_switch = (not can_main_attack and (bench_attacker or (active_id != Budew and field_counts[Budew] >= 1 and state.turn >= 2)))
    effect_card_id = 0 if select.effect == None else select.effect.id
    context_card_id = 0 if select.contextCard == None else select.contextCard.id
    
    scores = []  # Score for each action
    for o in select.option:
        score = 0  # The default and baseline score is 0.
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            if context == SelectContext.IS_FIRST:
                score = -1
            else:
                score = 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card != None:
                energy_count = 0
                hp = 0
                if isinstance(card, Pokemon):
                    energy_count = len(card.energies)
                    hp = card.hp
                if (context == SelectContext.SWITCH
                    or context == SelectContext.TO_ACTIVE
                    or context == SelectContext.SETUP_ACTIVE_POKEMON):
                    # Selection of the Pokémon to send to the Active Spot
                    if o.playerIndex == my_index:
                        if card.id == Dreepy:
                            score += 10000
                        elif card.id == Drakloak:
                            if energy_count >= 1:
                                score += 20000
                            else:
                                score -= 10000
                        elif card.id == Dragapult_ex:
                            score += 50000
                        elif card.id == Budew:
                            if context != SelectContext.SWITCH:
                                score += 100000
                            elif not bench_attacker:
                                score += 30000
                        elif card.id == Fezandipiti_ex:
                            score -= 1000
                        elif card.id == Meowth_ex:
                            score -= 2000
                    else:
                        if plan_a.attack == o.index + 1:
                            score += 100000
                    score += energy_count * 1000
                    score += hp
                elif context == SelectContext.SETUP_BENCH_POKEMON:
                    if my_index == state.firstPlayer or card.id != Dreepy:
                        score = -1
                elif context == SelectContext.TO_BENCH or context == SelectContext.TO_HAND:
                    score = hand_score(card.id, False)
                    hand_counts[card.id] += 1
                    if effect_card_id == Crispin:
                        # Reverse scoring
                        score = 100000 - hand_score(card.id, True)
                elif context == SelectContext.DISCARD:
                    hand_counts[card.id] -= 1
                    if card_table[card.id].cardType == CardType.SUPPORTER:
                        support_count -= 1
                    score = -hand_score(card.id, False)
                elif context == SelectContext.DAMAGE_COUNTER or context == SelectContext.DAMAGE_COUNTER_ANY:
                    if hp > 0:
                        score = 100000 - 10 * hp + pokemon_score(card, False)
                        if context == SelectContext.DAMAGE_COUNTER:
                            if 210 <= hp <= 230:
                                score += 20000 + hp * 20
                                if o.area == AreaType.ACTIVE:
                                    score += 10000
                            elif 40 <= hp <= 90:
                                score += 10000 + hp * 20
                            elif hp <= 30:
                                score += -10000 + hp * 20
                            if card.id == 133 or card.id == 351:
                                score += 30000
                        else:
                            index = o.index + 1
                            if index in plan_b.counter:
                                score += 100000
                            else:
                                remain_damage = select.remainDamageCounter * 10
                                if 210 <= hp <= 200 + remain_damage:
                                    score += 30000
                                elif 20 <= hp <= 60 + remain_damage:
                                    score += 10000
                                elif hp == 10:
                                    score -= 100000
                            if no_damage_counter(card):
                                score = -1
                elif context == SelectContext.ATTACH_FROM:
                    score = attach_score(context_card_id, card, o.area == AreaType.ACTIVE)
                    if card.id == Dragapult_ex:
                        score += 200
        elif o.type == OptionType.ENERGY_CARD or o.type == OptionType.ENERGY:
            # Discarding energy (Retreat or Crushing Hammer)
            if o.playerIndex != state.yourIndex:
                if o.area == AreaType.BENCH:
                    score = 20
                else:
                    score = 10
                card = get_card(obs, o.area, o.index, o.playerIndex)
                if card_table[card.id].cardType == CardType.SPECIAL_ENERGY:
                    score += 1
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            card_score = hand_scores[o.index]
            if card.id == Dreepy:
                score = 51000
            elif card.id == Fezandipiti_ex:
                if card_score > 0:
                    score = 53000
                else:
                    score = -1
            elif card.id == Latias_ex:
                if active_id != Drakloak and active_id != Dragapult_ex:
                    score = 51000
                else:
                    score = -1
            elif card.id == Budew:
                if field_counts[Budew] == 0 and field_counts[Dragapult_ex] == 0:
                    score = 52000
                else:
                    score = -1
            elif card.id == Meowth_ex:
                if state.supporterPlayed or stadium_id == Team_Rocket_Watchtower:
                    score = -1
                elif support_count == 0:
                    score = 50000
                elif support_count == hand_counts[Boss_Orders] and not plan_a.attack <= 0:
                    score = 50000
                else:
                    score = -1
            elif card.id == Rare_Candy:
                if no_more_dex:
                    score = -1
                else:
                    score = 75000
            elif card.id == Unfair_Stamp:
                score = 15000
            elif card.id == Night_Stretcher:
                if card_score >= 18000:
                    score = 42000
                else:
                    score = -1
            elif card.id == Crushing_Hammer:
                score = 40000
            elif card.id == Boss_Orders:
                if card.id == use_support:
                    score = 35000
                else:
                    score = -1
            elif card.id == Lillie_Determination:
                if card.id == use_support:
                    score = 14000
                else:
                    score = -1
            elif card.id == Team_Rocket_Watchtower:
                if stadium_id > 0 or state.turn == 1:
                    score = 80000
                else:
                    score = -1
            elif no_draw:
                score = -1
            elif card.id == Buddy_Buddy_Poffin:
                if deck_counts[Dreepy] > 0:
                    score = 46000
                else:
                    score = -1
            elif card.id == Ultra_Ball:
                if negative_hand_count >= 2:
                    score = 44000
                else:
                    score = -1
            elif card.id == Poke_Pad:
                if deck_counts[Dreepy] + deck_counts[Drakloak] > 0:
                    score = 45000
                else:
                    score = -1
            elif card.id == Crispin or card.id == Brock_Scouting:
                if card.id == use_support:
                    score = 35000
                else:
                    score = -1
        elif o.type == OptionType.ATTACH:
            card = get_card(obs, o.area, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = attach_score(card.id, pokemon, o.inPlayArea == AreaType.ACTIVE)
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score += len(pokemon.energies)
            if pokemon.id == Dreepy:
                score += 30000
            elif field_counts[Dragapult_ex] >= 2 or (field_counts[Dragapult_ex] == 1 and len(op_state.prize) <= 2):
                score = -1
            else:
                score += 70000
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if no_draw:
                score = -1
            elif card.id == 1267:  # Lumiose City
                score = 1
            else:
                score = 40000
        elif o.type == OptionType.RETREAT:
            if do_switch:
                score = 10000
            else:
                score = -1
        elif o.type == OptionType.ATTACK:
            score = o.attackId

        scores.append(score)

    output = []
    if len(scores) >= 1:
        # Select in descending order of score
        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        for i in range(select.maxCount):
            # If the score is negative, do not select it if skipping is possible
            if (sorted_scores[i][1] >= 0
                or select.minCount > i
                or (context != SelectContext.TO_BENCH and context != SelectContext.SETUP_BENCH_POKEMON)):
                output.append(sorted_scores[i][0])
                
    return output


        

## Opponent Bots


In [ ]:
%%writefile opp_abomasnow.py
import os
import sys
from collections import defaultdict

from cg.api import AreaType, CardType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

"""
Mega Abomasnow ex Deck (self-contained: deck is built inline, no deck.csv needed)
This is a simple deck that attacks with Hammer-lanche.
"""

# ── Decklist constants ──────────────────────────────────────────────────────
Kyogre = 721                 # x2
Snover = 722                 # x4
Mega_Abomasnow_ex = 723      # x4
Ultra_Ball = 1121            # x4
Precious_Trolley = 1126      # x1
Carmine = 1192               # x4
Lillie_Determination = 1227  # x4
Surfing_Beach = 1262         # x3
Basic_Water_Energy = 3       # x34

# ── Build the deck inline (reverse-engineered from the decklist above) ──────
DECK_RECIPE = [
    (Kyogre, 2),
    (Snover, 4),
    (Mega_Abomasnow_ex, 4),
    (Ultra_Ball, 4),
    (Precious_Trolley, 1),
    (Carmine, 4),
    (Lillie_Determination, 4),
    (Surfing_Beach, 3),
    (Basic_Water_Energy, 34),
]

def build_deck(recipe):
    deck = []
    for card_id, count in recipe:
        deck.extend([card_id] * count)
    return deck

my_deck = build_deck(DECK_RECIPE)
assert len(my_deck) == 60, f"Deck must be 60 cards, got {len(my_deck)}"

# ── Card metadata lookup (kept for parity with the original) ────────────────
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Helper function to safely extract a Card or Pokemon object from specific zones."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select == None:
        return my_deck

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]

    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    bench_attacker_index0 = -1  # Mega Abomasnow ex
    bench_attacker_index1 = -1  # Kyogre
    for i, card in enumerate(my_state.bench):
        field_counts[card.id] += 1
        if card.id == Mega_Abomasnow_ex and len(card.energies) >= 2:
            bench_attacker_index0 = i
        elif card.id == Kyogre and len(card.energies) >= 1:
            bench_attacker_index1 = i

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    op_active_hp = 0
    for card in state.players[1 - my_index].active:
        if card == None:
            continue
        op_active_hp = card.hp

    prefer_ky = op_active_hp <= 20 * discard_counts[Basic_Water_Energy]
    switch_index = -1
    for card in my_state.active:
        if card == None:
            continue
        field_counts[card.id] += 1
        if card.id == Mega_Abomasnow_ex and len(card.energies) >= 2:
            if prefer_ky and bench_attacker_index1 >= 0:
                switch_index = bench_attacker_index1
        elif card.id == Kyogre and len(card.energies) >= 1:
            if not prefer_ky and bench_attacker_index0 >= 0:
                switch_index = bench_attacker_index0
        elif bench_attacker_index0 >= 0:
            switch_index = bench_attacker_index0

    scores = []
    for o in select.option:
        score = 0
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            score = 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card != None:
                energy_count = 0
                if isinstance(card, Pokemon):
                    energy_count = len(card.energies)
                if (context == SelectContext.SWITCH
                    or context == SelectContext.TO_ACTIVE
                    or context == SelectContext.SETUP_ACTIVE_POKEMON):
                    score += energy_count * 2
                    if o.index == switch_index:
                        score += 100
                    if card.id == Mega_Abomasnow_ex:
                        score += 20
                    elif card.id == Kyogre:
                        score += 10
                elif context == SelectContext.TO_BENCH or context == SelectContext.TO_HAND:
                    if card.id == Snover:
                        if field_counts[card.id] >= 1:
                            score += 5
                        elif field_counts[Mega_Abomasnow_ex] >= 1:
                            score += 15
                        else:
                            score += 30
                    elif card.id == Mega_Abomasnow_ex:
                        if field_counts[Snover] >= 1 and field_counts[card.id] + hand_counts[card.id] == 0:
                            score += 100
                        else:
                            score += 10
                    elif card.id == Kyogre:
                        if field_counts[card.id] >= 1:
                            score += 1
                        else:
                            score += 20
                elif context == SelectContext.DISCARD:
                    if card.id == Basic_Water_Energy:
                        score += 100
                    elif card.id == Mega_Abomasnow_ex:
                        score += 10
                    elif card.id == Carmine:
                        if hand_counts[Lillie_Determination] >= 1:
                            score += 30
                    elif card.id == Lillie_Determination:
                        score -= 20
                    if hand_counts[card.id] >= 2:
                        score += 500
                    hand_counts[card.id] -= 1
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            score = 10000
            if card.id == Ultra_Ball:
                if hand_counts[Basic_Water_Energy] >= 3 or (my_state.handCount >= 4 and (field_counts[Mega_Abomasnow_ex] + hand_counts[Mega_Abomasnow_ex] == 0 or field_counts[Mega_Abomasnow_ex] + field_counts[Snover] == 0 or field_counts[Kyogre] == 0)):
                    score = 4000
                else:
                    score = -1
            elif card.id == Carmine:
                if field_counts[Snover] >= 1 and hand_counts[Mega_Abomasnow_ex] >= 1:
                    score = -1
                else:
                    score = 3000
            elif card.id == Lillie_Determination:
                if field_counts[Snover] >= 1 and field_counts[Mega_Abomasnow_ex] == 0 and hand_counts[Mega_Abomasnow_ex] >= 1:
                    score = -1
                else:
                    score = 3100
        elif o.type == OptionType.ATTACH:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 5000
            energy_count = len(pokemon.energies)
            if energy_count == 0:
                if o.inPlayArea == AreaType.BENCH:
                    score += 1
            if pokemon.id == Snover:
                score += 1
                if energy_count == 1:
                    score -= 100
                elif energy_count >= 2:
                    score -= 400
                if bench_attacker_index0 >= 0:
                    score -= 300
            elif pokemon.id == Mega_Abomasnow_ex:
                score += 10
                if energy_count == 1:
                    score += 30
                elif energy_count >= 2:
                    score -= 300
                if bench_attacker_index0 >= 0:
                    score -= 200
            elif pokemon.id == Kyogre:
                score += 5
                if len(pokemon.energies) >= 1:
                    score -= 200
                if bench_attacker_index1 >= 0:
                    score -= 200
            if o.inPlayArea == AreaType.ACTIVE:
                if bench_attacker_index0 >= 0 and bench_attacker_index1 >= 0 and energy_count <= 2:
                    score += 200
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 10000 + len(pokemon.energies)
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == Surfing_Beach and switch_index >= 0:
                score = 2000
            else:
                score = -1
        elif o.type == OptionType.RETREAT:
            if switch_index >= 0:
                score = 1500
            else:
                score = -1
        elif o.type == OptionType.ATTACK:
            score = 1000
            if o.attackId == 1042:  # Riptide
                score += discard_counts[Basic_Water_Energy] * 20 - 90
            elif o.attackId == 1046:  # Hammer-lanche
                if op_active_hp <= 200:
                    score -= 100
                else:
                    score += 100

        scores.append(score)

    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    return desc_indices[:select.maxCount]

In [ ]:
%%writefile opp_iono.py
import os
import sys
from collections import defaultdict

from cg.api import AreaType, CardType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

"""
Iono's Deck (self-contained: deck built inline, no deck.csv needed)
Intermediate Level
This deck aims to load up your Pokémon with as much Energy as possible to unleash a powerful Voltaic Chain.
"""

# ── Decklist constants ──────────────────────────────────────────────────────
Iono_Voltorb = 265           # x3
Iono_Tadbulb = 268           # x3
Iono_Bellibolt_ex = 269      # x3
Iono_Wattrel = 270           # x3
Iono_Kilowattrel = 271       # x3
Buddy_Buddy_Poffin = 1086    # x3
Night_Stretcher = 1097       # x2
Max_Rod = 1110               # x1
Energy_Retrieval = 1118      # x1
Ultra_Ball = 1121            # x3
Poke_Pad = 1152              # x2
Lillie_Determination = 1227  # x4
Canari = 1233                # x4
Levincia = 1254              # x3
Basic_Lightning_Energy = 4   # x22

# ── Build the deck inline (reverse-engineered from the decklist above) ──────
DECK_RECIPE = [
    (Iono_Voltorb, 3),
    (Iono_Tadbulb, 3),
    (Iono_Bellibolt_ex, 3),
    (Iono_Wattrel, 3),
    (Iono_Kilowattrel, 3),
    (Buddy_Buddy_Poffin, 3),
    (Night_Stretcher, 2),
    (Max_Rod, 1),
    (Energy_Retrieval, 1),
    (Ultra_Ball, 3),
    (Poke_Pad, 2),
    (Lillie_Determination, 4),
    (Canari, 4),
    (Levincia, 3),
    (Basic_Lightning_Energy, 22),
]

def build_deck(recipe):
    deck = []
    for card_id, count in recipe:
        deck.extend([card_id] * count)
    return deck

my_deck = build_deck(DECK_RECIPE)
assert len(my_deck) == 60, f"Deck must be 60 cards, got {len(my_deck)}"

# ── Card metadata lookup ────────────────────────────────────────────────────
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}

can_attack = False

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Helper function to safely extract a Card or Pokemon object from specific zones."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def agent(obs_dict: dict) -> list[int]:
    """Main Agent Function.

    Each element in the returned list must be >= 0 and < len(obs.select.option).
    The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.

    Returns:
        list[int]: A list of option index.
    """
    obs = to_observation_class(obs_dict)
    if obs.select == None:
        return my_deck

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
    op_prize = len(op_state.prize)

    field_counts = defaultdict(int)
    field_hand_counts = defaultdict(int)
    active_attacker = False
    bench_attacker = False

    energy_count = 0
    can_ability = False
    for p in my_state.active:
        if p == None:
            continue
        field_counts[p.id] += 1
        field_hand_counts[p.id] += 1
        energy_count += len(p.energies)
        if p.id == Iono_Kilowattrel:
            if len(p.energies) > 0:
                can_ability = True
        if p.id == Iono_Voltorb:
            if len(p.energies) >= 2:
                active_attacker = True
    for p in my_state.bench:
        field_counts[p.id] += 1
        field_hand_counts[p.id] += 1
        energy_count += len(p.energies)
        if p.id == Iono_Kilowattrel:
            if len(p.energies) > 0:
                can_ability = True
        if p.id == Iono_Voltorb:
            if len(p.energies) >= 2:
                bench_attacker = True
    field_pokemon1 = field_counts[Iono_Tadbulb] + field_counts[Iono_Bellibolt_ex]
    field_pokemon2 = field_counts[Iono_Wattrel] + field_counts[Iono_Kilowattrel]
    no_more_pokemon = (len(my_state.bench) >= 5)
    if field_counts[Iono_Tadbulb] + field_counts[Iono_Wattrel] >= 1:
        no_more_pokemon = False

    stadium_id = 0
    for c in state.stadium:
        stadium_id = c.id

    hand_counts = defaultdict(int)
    hand_scores = []
    unused_hand_count = 0
    for c in my_state.hand:
        data = card_table[c.id]

        score = -10000
        if c.id == Iono_Voltorb:
            score = 100
        elif c.id == Iono_Bellibolt_ex:
            if field_counts[c.id] <= 1:
                score = 120
        elif c.id == Iono_Kilowattrel:
            if field_counts[c.id] <= 1:
                score = 140
        elif c.id == Ultra_Ball:
            if not no_more_pokemon:
                score = 10
        elif c.id == Night_Stretcher:
            score = 50
        elif c.id == Energy_Retrieval:
            score = 20
        elif c.id == Max_Rod:
            score = 1000
        elif c.id == Lillie_Determination:
            score = 150
        elif c.id == Canari:
            score = 160
        elif c.id == Levincia:
            if stadium_id != Levincia:
                score = 30
        elif c.id == Basic_Lightning_Energy:
            score = -10
        score -= hand_counts[c.id] * 100
        hand_scores.append(score)
        if score < 0:
            unused_hand_count += 1

        hand_counts[c.id] += 1
        field_hand_counts[c.id] += 1

    discard_counts = defaultdict(int)
    for c in my_state.discard:
        discard_counts[c.id] += 1

    global can_attack
    if context == SelectContext.MAIN:
        can_attack = False
        for o in select.option:
            if o.type == OptionType.ATTACK:
                can_attack = True

    op_active_hp = 10000
    if len(op_state.active) >= 1:
        if op_state.active[0] != None:
            op_active_hp = op_state.active[0].hp

    no_draw = (my_state.deckCount <= 5)

    scores = []
    id_counts = defaultdict(int)
    for o in select.option:
        score = 0
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            score = 1
        elif o.type == OptionType.ATTACH or context == SelectContext.ATTACH_FROM:
            if o.type == OptionType.ATTACH:
                p = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            else:
                p = get_card(obs, o.area, o.index, o.playerIndex)
            score = 40000
            if p.id == Iono_Voltorb:
                if len(p.energies) >= 2:
                    if o.inPlayArea == AreaType.ACTIVE and not can_attack:
                        score += 3000
                else:
                    if o.inPlayArea == AreaType.ACTIVE:
                        score += 5000
                    elif bench_attacker or active_attacker:
                        score += 100
                    else:
                        score += 1000
            elif p.id == Iono_Tadbulb:
                score += 10 - len(p.energies)
            elif p.id == Iono_Bellibolt_ex:
                if len(p.energies) >= 4:
                    if o.inPlayArea == AreaType.ACTIVE and not can_attack:
                        score += 500
                else:
                    if o.inPlayArea == AreaType.ACTIVE:
                        score += 800
                    elif bench_attacker or active_attacker:
                        score += 14 - len(p.energies)
                    else:
                        score += 100
            elif p.id == Iono_Wattrel:
                if len(p.energies) >= 1 or o.inPlayArea == AreaType.ACTIVE:
                    score += 10 - len(p.energies)
                else:
                    score += 6000
            elif p.id == Iono_Kilowattrel:
                if len(p.energies) >= 1:
                    score += 11 - len(p.energies)
                else:
                    score += 8000
        elif o.type == OptionType.CARD:
            c = get_card(obs, o.area, o.index, o.playerIndex)
            if c != None:
                data = card_table[c.id]
                if (context == SelectContext.SWITCH
                    or context == SelectContext.TO_ACTIVE
                    or context == SelectContext.SETUP_ACTIVE_POKEMON):
                    energy = 0
                    if isinstance(c, Pokemon):
                        energy = len(c.energies)
                        score -= c.hp
                        score -= energy * 100
                    if c.id == Iono_Voltorb:
                        if 20 + energy_count * 20 >= op_active_hp:
                            score += 100000
                        else:
                            score += 1500
                        if energy >= 1:
                            score += 200
                            if energy >= 2:
                                score += 10000
                    elif c.id == Iono_Bellibolt_ex:
                        score += 1000
                        if energy >= 4:
                            score += 1000
                    elif c.id == Iono_Tadbulb:
                        score += 10
                elif context == SelectContext.TO_HAND or context == SelectContext.TO_BENCH:
                    if c.id == Basic_Lightning_Energy:
                        score += 1
                    elif c.id == Iono_Voltorb:
                        if o.area == AreaType.DISCARD:
                            score += 100000
                        if field_counts[c.id] == 0:
                            score += 110
                        elif field_counts[c.id] == 1 and op_prize >= 2:
                            score += 5
                    elif c.id == Iono_Tadbulb:
                        if field_pokemon1 == 0:
                            score += 200
                        elif field_pokemon1 == 1:
                            if op_prize >= 3 or (op_prize >= 2 and field_counts[Iono_Bellibolt_ex] == 0):
                                score += 20
                    elif c.id == Iono_Bellibolt_ex:
                        if field_hand_counts[c.id] == 0:
                            score += 250
                            if field_counts[Iono_Tadbulb] > 0:
                                score += 300
                        elif field_hand_counts[c.id] == 1:
                            if op_prize >= 3:
                                score += 30
                                if field_counts[Iono_Tadbulb] > 0:
                                    score += 30
                    elif c.id == Iono_Wattrel:
                        if field_pokemon2 == 0:
                            score += 320
                        elif field_pokemon2 == 1:
                            score += 15
                    elif c.id == Iono_Kilowattrel:
                        if field_hand_counts[c.id] == 0:
                            score += 300
                            if field_counts[Iono_Wattrel] > 0:
                                score += 250
                        elif field_hand_counts[c.id] == 1:
                            score += 25
                            if field_counts[Iono_Wattrel] > 0:
                                score += 25

                    if c.id != Basic_Lightning_Energy:
                        if hand_counts[c.id] >= 2:
                            score -= 20000
                        elif hand_counts[c.id] >= 1:
                            score -= 2000
                        if id_counts[c.id] == 1:
                            score -= 1000
                        elif id_counts[c.id] >= 2:
                            score -= 10000

                    id_counts[c.id] += 1
                elif context == SelectContext.DISCARD:
                    if o.area == AreaType.HAND and o.playerIndex == my_index:
                        score = -hand_scores[o.index]
        elif o.type == OptionType.PLAY:
            c = get_card(obs, AreaType.HAND, o.index, my_index)
            data = card_table[c.id]
            if data.cardType == CardType.STADIUM:
                if discard_counts[Basic_Lightning_Energy] >= 1 or can_ability:
                    score = 85000
                else:
                    score = -1
            elif data.cardType == CardType.SUPPORTER:
                score = 25000
                if c.id == Lillie_Determination:
                    score += 1000
                elif no_draw:
                    score = -1
                elif c.id == Canari:
                    if no_more_pokemon:
                        score = -1
                    elif field_counts[Iono_Voltorb] > 0 and field_counts[Iono_Bellibolt_ex] > 0 and field_counts[Iono_Kilowattrel] > 0:
                        score += 100
                    else:
                        score += 2000
            elif data.cardType == CardType.POKEMON:
                score = 100000
                if c.id == Iono_Voltorb and field_counts[Iono_Voltorb] >= 2:
                    score = -1
                elif c.id == Iono_Tadbulb and field_pokemon1 >= 2:
                    score = -1
                elif c.id == Iono_Wattrel and field_pokemon2 >= 2:
                    if op_prize >= 2 or field_counts[Iono_Voltorb] == 0 or field_counts[Iono_Bellibolt_ex] == 0:
                        score = -1
            else:
                if c.id == Night_Stretcher:
                    if discard_counts[Iono_Voltorb] > 0 or (discard_counts[Iono_Bellibolt_ex] > 0 and field_counts[Iono_Tadbulb] > 0) or (discard_counts[Iono_Kilowattrel] > 0 and field_counts[Iono_Wattrel] > 0):
                        score = 75000
                    else:
                        score = -1
                elif c.id == Energy_Retrieval:
                    score = 61000
                elif c.id == Max_Rod:
                    if state.turn >= 3 and discard_counts[Basic_Lightning_Energy] >= 2:
                        score = 55000
                    else:
                        score = -1
                elif no_draw:
                    score = -1
                elif c.id == Buddy_Buddy_Poffin:
                    score = 80000
                elif c.id == Ultra_Ball:
                    if no_more_pokemon or state.turn <= 2:
                        score = -1
                    elif field_hand_counts[Iono_Bellibolt_ex] > 0 and field_hand_counts[Iono_Kilowattrel] > 0:
                        if unused_hand_count >= 2:
                            score = 45000
                        else:
                            score = -1
                    else:
                        if unused_hand_count >= 1:
                            score = 62000
                        else:
                            score = -1
                elif c.id == Poke_Pad:
                    score = 79000
        elif o.type == OptionType.EVOLVE:
            score = 110000
        elif o.type == OptionType.ABILITY:
            score = -1
            c = get_card(obs, o.area, o.index, my_index)
            if c.id == Iono_Bellibolt_ex:
                score = 50000
            elif c.id == Levincia:
                score = 8000
            elif not no_draw and c.id == Iono_Kilowattrel:
                score = 30000
        elif o.type == OptionType.RETREAT:
            if bench_attacker and not active_attacker:
                score = 10000
            else:
                score = -1
        elif o.type == OptionType.ATTACK:
            score = o.attackId

        scores.append(score)

    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    return desc_indices[:select.maxCount]

In [ ]:
%%writefile opp_lucario.py
from __future__ import annotations

from collections import defaultdict

from cg.api import (
    AreaType,
    Card,
    CardType,
    EnergyType,
    Observation,
    OptionType,
    Pokemon,
    SelectContext,
    all_card_data,
    to_observation_class,
)

class C:
    KYOGRE = 721
    SNOVER = 722
    MEGA_ABOMASNOW_EX = 723

    ABRA = 741
    KADABRA = 742
    ALAKAZAM = 743

    MAKUHITA = 673
    HARIYAMA = 674
    LUNATONE = 675
    SOLROCK = 676
    RIOLU = 677
    MEGA_LUCARIO_EX = 678
    DWEBBLE = 344
    CRUSTLE = 345

    BASIC_FIGHTING_ENERGY = 6
    DUSK_BALL = 1102
    SWITCH = 1123
    PREMIUM_POWER_PRO = 1141
    FIGHTING_GONG = 1142
    POKE_PAD = 1152
    HERO_CAPE = 1159
    BOSS_ORDERS = 1182
    CARMINE = 1192
    LILLIE_DETERMINATION = 1227
    GRAVITY_MOUNTAIN = 1252

    LUMIOSE_CITY = 1267
    LILLIES_PEARL = 1172
    LEGACY_ENERGY = 12

MEGA_BRAVE = 983
LOW_DECK_COUNT = 10

_ABRA_BONUS = 400
_KADABRA_BONUS = 400

# ── Build the deck inline (reverse-engineered from lucario_deck.csv) ─────────
LUCARIO_DECK_RECIPE = [
    (C.MAKUHITA, 2),
    (C.HARIYAMA, 2),
    (C.LUNATONE, 2),
    (C.SOLROCK, 3),
    (C.RIOLU, 4),                
    (C.MEGA_LUCARIO_EX, 4),
    (C.DUSK_BALL, 4),
    (C.SWITCH, 2),
    (C.PREMIUM_POWER_PRO, 4),
    (C.FIGHTING_GONG, 4),
    (C.POKE_PAD, 2),              
    (C.HERO_CAPE, 1),             
    (C.BOSS_ORDERS, 3),           
    (C.CARMINE, 4),
    (C.LILLIE_DETERMINATION, 4),
    (C.GRAVITY_MOUNTAIN, 1),     
    (C.BASIC_FIGHTING_ENERGY, 14), 
]

def build_deck(recipe):
    deck = []
    for card_id, count in recipe:
        deck.extend([card_id] * count)
    return deck

my_deck = build_deck(LUCARIO_DECK_RECIPE)
assert len(my_deck) == 60, f"Deck must be 60 cards, got {len(my_deck)}"

all_card = all_card_data()
card_table = {card.cardId: card for card in all_card}

class AttackPlan:
    def __init__(
        self,
        attacker: int = -1,
        target: int = -1,
        attack_index: int = -1,
        remain_hp: int = -1,
        needs_energy: bool = False,
    ):
        self.attacker = attacker
        self.target = target
        self.attack_index = attack_index
        self.remain_hp = remain_hp
        self.needs_energy = needs_energy

plan = AttackPlan()
pre_turn = -1
ability_used = False

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    player = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return player.hand[index]
        case AreaType.DISCARD:
            return player.discard[index]
        case AreaType.ACTIVE:
            return player.active[index]
        case AreaType.BENCH:
            return player.bench[index]
        case AreaType.PRIZE:
            return player.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def prize_count(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == C.LEGACY_ENERGY:
            count -= 1
    for card in pokemon.tools:
        if card.id == C.LILLIES_PEARL and "Lillie" in data.name:
            count -= 1
    return max(0, count)

def target_score(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    score = prize_count(pokemon) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130

    if pokemon.id in {144, 322, 323, 337}:
        score -= 200
    if pokemon.id == C.SNOVER:
        score += 950
    elif pokemon.id == C.MEGA_ABOMASNOW_EX:
        score += 250
    if pokemon.id == C.ABRA:
        score += _ABRA_BONUS
    elif pokemon.id == C.KADABRA:
        score += _KADABRA_BONUS
    if pokemon.id == C.RIOLU:
        score += 800
    elif pokemon.id == C.MEGA_LUCARIO_EX:
        score += 100
    if pokemon.id == 112 and len(pokemon.energies) >= 1:
        score += 300
    score += pokemon.hp
    return score

class LucarioPolicy:
    def __init__(self, obs: Observation):
        self.obs = obs
        self.state = obs.current
        self.select = obs.select
        self.context = self.select.context
        self.my_index = self.state.yourIndex
        self.op_index = 1 - self.my_index
        self.me = self.state.players[self.my_index]
        self.opponent = self.state.players[self.op_index]
        self.my_prizes_left = len(self.me.prize)

        self.field_counts = defaultdict(int)
        self.hand_counts = defaultdict(int)
        self.discard_counts = defaultdict(int)
        self.has_ready_lucario_line = False
        self.has_ready_hariyama_line = False
        self.can_switch = False
        self.can_gust = False
        self.can_attack = False
        self.can_use_mega_brave = False
        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0

        self._count_cards()
        self._scan_main_options()

    def choose(self) -> list[int]:
        if not self.select.option or self.select.maxCount == 0:
            return []

        if self.context == SelectContext.MAIN:
            self._plan_attack()

        scores = [self._score_option(option) for option in self.select.option]
        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]
        self._remember_lunatone_ability(ranked)
        return ranked[: self.select.maxCount]

    def _count_cards(self) -> None:
        for pokemon in self.me.active + self.me.bench:
            if pokemon is None:
                continue
            self.field_counts[pokemon.id] += 1
            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3:
                self.has_ready_hariyama_line = True
            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2:
                self.has_ready_lucario_line = True

        for card in self.me.hand:
            self.hand_counts[card.id] += 1
        for card in self.me.discard:
            self.discard_counts[card.id] += 1

    def _scan_main_options(self) -> None:
        if self.context != SelectContext.MAIN:
            return
        for option in self.select.option:
            if option.type == OptionType.PLAY:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.SWITCH:
                    self.can_switch = True
                elif card.id == C.BOSS_ORDERS:
                    self.can_gust = True
            elif option.type == OptionType.EVOLVE:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.HARIYAMA:
                    self.can_gust = True
            elif option.type == OptionType.RETREAT:
                self.can_switch = True
            elif option.type == OptionType.ATTACK:
                self.can_attack = True
                if option.attackId == MEGA_BRAVE:
                    self.can_use_mega_brave = True

    def _my_board(self) -> list[Pokemon | None]:
        return self.me.active + self.me.bench

    def _opponent_board(self) -> list[Pokemon | None]:
        return self.opponent.active + self.opponent.bench

    def _opponent_has_crustle_axis(self) -> bool:
        return any(
            pokemon is not None and pokemon.id in {C.DWEBBLE, C.CRUSTLE}
            for pokemon in self._opponent_board()
        )

    def _opponent_is_water_deck(self) -> bool:
        return any(
            pokemon is not None and pokemon.id in {C.KYOGRE, C.SNOVER, C.MEGA_ABOMASNOW_EX}
            for pokemon in self._opponent_board()
        )

    def _should_preserve_hariyama(self) -> bool:
        return (
            self._opponent_has_crustle_axis()
            and self.hand_counts[C.HARIYAMA] >= 1
            and any(pokemon is not None and pokemon.id == C.MAKUHITA for pokemon in self._my_board())
        )

    def _can_evolve_board_index(self, board_index: int) -> bool:
        for option in self.select.option:
            if option.type != OptionType.EVOLVE:
                continue
            target_index = option.inPlayIndex
            if option.inPlayArea == AreaType.BENCH:
                target_index += 1
            if target_index == board_index:
                return True
        return False

    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:
        energy_required = 0
        base_damage = 0
        base_score = 0

        if pokemon.id == C.MEGA_LUCARIO_EX:
            if attack_index == 0:
                energy_required = 1
                base_damage = 130
                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])
            else:
                energy_required = 2
                base_damage = 270
            if self.my_prizes_left in {2, 3}:
                base_score -= 500
        elif attack_index == 1:
            return None
        elif pokemon.id == C.HARIYAMA:
            energy_required = 3
            base_damage = 210
        elif pokemon.id == C.MAKUHITA:
            return None
        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1:
            energy_required = 1
            base_damage = 70

        if base_damage <= 0:
            return None
        return energy_required, base_damage, base_score

    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):
        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index):
            return 3, 210, -100
        return self._base_attack(pokemon, attack_index)

    def _plan_attack(self) -> None:
        global plan
        best_score = -1
        plan = AttackPlan()

        if self.state.turn < 2:
            return

        for attacker_index, my_pokemon in enumerate(self._my_board()):
            if my_pokemon is None:
                continue
            if attacker_index != 0 and not self.can_switch:
                break

            for attack_index in range(2):
                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)
                if attack is None:
                    continue
                energy_required, base_damage, base_score = attack

                energy_count = len(my_pokemon.energies)
                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave:
                    break

                needs_energy = False
                if energy_count < energy_required:
                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:
                        energy_count += 1
                        needs_energy = energy_count >= energy_required
                    if not needs_energy:
                        continue

                for target_index, op_pokemon in enumerate(self._opponent_board()):
                    if op_pokemon is None:
                        continue
                    if target_index != 0 and not self.can_gust:
                        break

                    damage = base_damage
                    if my_pokemon.id == C.MEGA_LUCARIO_EX and op_pokemon.id == C.CRUSTLE:
                        damage = 0
                    else:
                        op_data = card_table[op_pokemon.id]
                        if op_data.weakness == EnergyType.FIGHTING:
                            damage *= 2
                        elif op_data.resistance == EnergyType.FIGHTING:
                            damage -= 30

                    score = target_score(op_pokemon)
                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0
                    if prize == 0:
                        score *= damage / op_pokemon.hp
                    if len(self.opponent.prize) <= prize:
                        score = 50000

                    score += base_score
                    score += 220 if attacker_index == 0 else 0
                    score += 300 if target_index == 0 else 0
                    score += energy_count

                    if score > best_score:
                        best_score = score
                        plan = AttackPlan(
                            attacker=attacker_index,
                            target=target_index,
                            attack_index=attack_index,
                            remain_hp=op_pokemon.hp - damage,
                            needs_energy=needs_energy,
                        )

    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        score = 8000 + (10 if active else 0)

        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
            score += 1 if pokemon.id == C.HARIYAMA else 0
            score += 100 if energy_count < 3 else 0
            score -= 50 if self.has_ready_hariyama_line else 0
        elif pokemon.id == C.LUNATONE:
            score -= 100
        elif pokemon.id == C.SOLROCK:
            score += 20 if energy_count < 1 else -100
        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:
            score += 1 if pokemon.id == C.MEGA_LUCARIO_EX else 0
            score += 100 if energy_count < 2 else 0
            score -= 50 if self.has_ready_lucario_line else 0
        return score

    def _score_option(self, option) -> float:
        if option.type == OptionType.NUMBER:
            return option.number
        if option.type == OptionType.YES:
            return 100 if self.context == SelectContext.IS_FIRST else 1
        if option.type == OptionType.NO:
            return 0
        if option.type == OptionType.CARD:
            return self._score_card_choice(option)
        if option.type == OptionType.PLAY:
            return self._score_play(option)
        if option.type == OptionType.ATTACH:
            return self._score_attach(option)
        if option.type == OptionType.EVOLVE:
            return self._score_evolve(option)
        if option.type == OptionType.ABILITY:
            return self._score_ability(option)
        if option.type == OptionType.RETREAT:
            return 2000 if plan.attacker >= 1 else -1
        if option.type == OptionType.ATTACK:
            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000
        return 0

    def _score_card_choice(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, option.playerIndex)
        if card is None:
            return 0

        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:
            return self._score_active_choice(option, card)
        if self.context == SelectContext.SETUP_ACTIVE_POKEMON:
            return self._score_setup_active(card)
        if self.context == SelectContext.TO_HAND:
            return self._score_to_hand(card)
        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon):
            return self._energy_target_score(card, option.area == AreaType.ACTIVE)
        return 0

    def _score_active_choice(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0

        if option.playerIndex != self.my_index:
            return 100 if option.index == plan.target - 1 else 0

        score = len(card.energies) * 2
        if option.index == plan.attacker - 1:
            score += 100
        if card.id == C.MEGA_LUCARIO_EX:
            score += 8 if self.my_prizes_left in {2, 3} else 20
        elif card.id == C.HARIYAMA and len(card.energies) >= 2:
            score += 15
        elif card.id == C.MAKUHITA and len(card.energies) >= 2:
            score += 10
        elif card.id == C.SOLROCK:
            score += 5
        elif card.id == C.RIOLU:
            score += 4
        return score

    def _score_setup_active(self, card: Pokemon | Card) -> int:
        if card.id == C.SOLROCK:
            return 2 if self.state.firstPlayer == self.my_index else 4
        if card.id == C.RIOLU:
            return 3
        if card.id == C.MAKUHITA:
            return 1
        return 0

    def _score_to_hand(self, card: Pokemon | Card) -> float:
        score = 200 - self.hand_counts[card.id] * 100
        if card.id == C.MAKUHITA:
            score += -10 if self.field_counts[card.id] >= 1 else 10
        elif card.id == C.HARIYAMA:
            score += 20 if self.field_counts[C.MAKUHITA] >= 1 else -20
        elif card.id == C.LUNATONE:
            score += -250 if self.field_counts[card.id] >= 1 else 60
        elif card.id == C.SOLROCK:
            score += -250 if self.field_counts[card.id] >= 1 else 50
        elif card.id == C.RIOLU:
            lucario_line = self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX]
            score += -150 if lucario_line >= 2 else -3 if lucario_line >= 1 else 40
        elif card.id == C.MEGA_LUCARIO_EX:
            score += 40 if self.field_counts[C.RIOLU] >= 1 else -15
        elif card.id == C.BASIC_FIGHTING_ENERGY:
            score += 30 if not ability_used or not self.state.energyAttached else -1
        return score

    def _score_play(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        data = card_table[card.id]
        if data.cardType == CardType.POKEMON:
            return self._score_play_pokemon(card)
        return self._score_play_trainer(card)

    def _score_play_pokemon(self, card: Card) -> float:
        score = 20000
        if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1:
            return -1
        if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2:
            return -1
        return score

    def _score_play_trainer(self, card: Card) -> float:
        if card.id == C.SWITCH:
            return 6000 if plan.attacker > 0 else -1
        if card.id == C.PREMIUM_POWER_PRO:
            if self.state.supporterPlayed and plan.remain_hp <= 0:
                return -1
            if not self.can_attack:
                can_bridge_draw = (
                    not self.state.supporterPlayed
                    and self.hand_counts[C.CARMINE] > 0
                    and self.hand_counts[C.LILLIE_DETERMINATION] == 0
                    and not self._low_deck()
                )
                return 3050 if can_bridge_draw else -1
            return 5000
        if card.id == C.BOSS_ORDERS:
            return 3200 if plan.target >= 1 else -1
        if card.id == C.CARMINE:
            if self._should_preserve_hariyama():
                return -1
            return -1 if self._low_deck() else 3000
        if card.id == C.LILLIE_DETERMINATION:
            return -1 if self._low_deck() else 3100
        if card.id == C.GRAVITY_MOUNTAIN:
            return self._score_gravity_mountain()
        return 10000

    def _score_gravity_mountain(self) -> float:
        opponent_has_stage2 = any(
            pokemon is not None and card_table[pokemon.id].stage2 for pokemon in self._opponent_board()
        )
        if opponent_has_stage2:
            return 3500
        return 1200 if self.stadium_id else -1

    def _low_deck(self) -> bool:
        return self.me.deckCount <= LOW_DECK_COUNT

    def _score_attach(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon):
            return 0

        if card.id == C.HERO_CAPE:
            score = 7000
            if self._opponent_is_water_deck():
                if pokemon.id == C.RIOLU:
                    return 12200
                if pokemon.id == C.MEGA_LUCARIO_EX:
                    return 12800
            if pokemon.id == C.RIOLU:
                score += 100
            elif pokemon.id == C.MEGA_LUCARIO_EX:
                score += 200
            return score

        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)
        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1
        if board_index == plan.attacker and plan.needs_energy:
            score += 200
        return score

    def _score_evolve(self, option) -> float:
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon):
            return 0
        evolved = get_card(self.obs, option.area, option.index, self.my_index)
        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1
        if pokemon.id == C.MAKUHITA and plan.target == 0 and not (
            evolved is not None and evolved.id == C.HARIYAMA and board_index == plan.attacker
        ):
            return -1
        return 9000 + len(pokemon.energies)

    def _score_ability(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card.id == C.LUMIOSE_CITY:
            return 1
        if card.id == C.LUNATONE and self._low_deck():
            return -1
        return 30000

    def _remember_lunatone_ability(self, ranked: list[int]) -> None:
        global ability_used
        if self.context != SelectContext.MAIN or not ranked:
            return
        option = self.select.option[ranked[0]]
        if option.type != OptionType.ABILITY:
            return
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card is not None and card.id == C.LUNATONE:
            ability_used = True

def agent(obs_dict: dict) -> list[int]:
    global pre_turn
    global ability_used
    global plan

    if obs_dict.get("select") is None and "current" not in obs_dict:
        pre_turn = -1
        ability_used = False
        plan = AttackPlan()
        return my_deck

    obs = to_observation_class(obs_dict)
    if obs.select is None:
        pre_turn = -1
        ability_used = False
        plan = AttackPlan()
        return my_deck

    if pre_turn != obs.current.turn:
        pre_turn = obs.current.turn
        ability_used = False
        plan = AttackPlan()

    return LucarioPolicy(obs).choose()

In [ ]:
%%writefile opp_archaludon.py
"""Archaludon ex + Cinderace — Rule-based agent (Public version)

Deck Concept:
  Cinderace's Explosiveness places it face-down as Active during setup.
  Turn 1 Turbo Flare ({C}=50) accelerates up to 3 Basic Energy from deck
  to benched Duraludon. Evolving into Archaludon ex triggers Assemble Alloy,
  attaching up to 2 Basic Metal Energy from discard to Metal Pokemon.
  Metal Defender ({M}{M}{M}=220) is the main attack; no Weakness next turn.
  Duraludon can attack directly with Raging Hammer ({M}{M}{C}=80 + 10 per
  damage counter) without evolving. Relicanth's Memory Dive also unlocks
  Raging Hammer on Archaludon ex after evolution. Hero's Cape gives +100 HP
  (HP400). Full Metal Lab reduces attack damage to Metal Pokemon by 30.

Pokemon:
  Duraludon (169)      - Basic Metal HP130. Hammer In {M}=30.
                         Raging Hammer {M}{M}{C}=80+10*damage_counters.
  Archaludon ex (190)  - Stage 1 from Duraludon, HP300. Assemble Alloy: on evolve
                         from hand, attach up to 2 Metal Energy from discard.
                         Metal Defender {M}{M}{M}=220, no Weakness next turn.
  Cinderace (666)      - Stage 2 HP160. Explosiveness: place face-down as Active
                         in setup from opening hand. Turbo Flare {C}=50, attach
                         up to 3 Basic Energy from deck to benched Pokemon.
  Relicanth (57)       - Basic HP100. Memory Dive: evolved Pokemon can use attacks
                         from previous Evolutions. Archaludon ex -> Raging Hammer.

Trainers:
  Poke Pad (1152), Ultra Ball (1121), Pokegear 3.0 (1122), Night Stretcher (1097),
  Jumbo Ice Cream (1147), Hero's Cape (1159), Boss's Orders (1182),
  Explorer's Guidance (1185), Lillie's Determination (1227), Full Metal Lab (1244) x4.

Energy: Basic Metal Energy (8) x11

Score system:
  Setup/play/evolve/attach: 1000~28000 (high = do first)
  Attack: damage value (always last — attacking ends the turn)
  Negative = skip if above minCount
"""

import os
import random
import sys

try:
    ROOT = __file__
except NameError:
    ROOT = None
CG_PATH = "/kaggle_simulations/agent"
for p in ([os.path.dirname(os.path.abspath(ROOT))] if ROOT else []) + [CG_PATH]:
    if p and p not in sys.path and os.path.isdir(p):
        sys.path.insert(0, p)

from cg.api import (
    AreaType,
    LogType,
    OptionType,
    SelectContext,
    all_card_data,
    to_observation_class,
)

try:
    from cg.api import all_attack
    ALL_ATTACKS = {a.attackId: a for a in all_attack()}
except Exception:
    ALL_ATTACKS = {}

# ── Card IDs ──

DURALUDON = 169
ARCHALUDON_EX = 190
CINDERACE = 666
RELICANTH = 57
CRUSTLE_LINE = {344, 345, 532}
STARMIE_LINE = {1030, 1031}
LUCARIO_LINE = {677, 678}
HOP_LINE = {288, 289, 299, 304, 307, 308, 309, 310, 878, 879}
HOP_SNORLAX = 304

METAL_ENERGY = 8

POKE_PAD = 1152
ULTRA_BALL = 1121
POKEGEAR = 1122
NIGHT_STRETCHER = 1097
JUMBO_ICE_CREAM = 1147
HERO_CAPE = 1159
BOSS = 1182
EXPLORER = 1185
LILLIE = 1227
FULL_METAL_LAB = 1244

# ── Build the deck inline (from deck.csv) ────────────────────────────────────
DECK_RECIPE = [
    (DURALUDON, 4),
    (ARCHALUDON_EX, 4),
    (CINDERACE, 4),
    (RELICANTH, 1),
    (POKE_PAD, 4),
    (ULTRA_BALL, 4),
    (POKEGEAR, 4),
    (NIGHT_STRETCHER, 3),
    (JUMBO_ICE_CREAM, 4),
    (HERO_CAPE, 1),
    (BOSS, 4),
    (EXPLORER, 4),
    (LILLIE, 4),
    (FULL_METAL_LAB, 4),
    (METAL_ENERGY, 11),
]

def build_deck(recipe):
    deck = []
    for card_id, count in recipe:
        deck.extend([card_id] * count)
    return deck

my_deck = build_deck(DECK_RECIPE)
assert len(my_deck) == 60, f"Deck must be 60 cards, got {len(my_deck)}"


RAGING_HAMMER = 224
METAL_DEFENDER = 253

_ATTACK_BASE_DMG = {METAL_DEFENDER: 220, 965: 50, 223: 30, 61: 30}

_SETUP_ACTIVE_PRIORITY = {
    CINDERACE: (100000, "Active: Cinderace Explosiveness"),
    DURALUDON: (20000, "Active fallback: Duraludon"),
    RELICANTH: (5000, "Active fallback: Relicanth"),
}

ALWAYS_SAFE_DISCARD = {METAL_ENERGY, CINDERACE}

CARD_DB = {c.cardId: c for c in all_card_data()}

MEGA_BRAVE = 983
PREMIUM_POWER_PRO = 1141
HARIYAMA_LINE = {673, 674}

# Track opponent's last-turn attack via logs
_opp_last_attack_id = None
_cur_turn_logs = []


def _update_opp_attack_tracking(obs):
    global _opp_last_attack_id, _cur_turn_logs
    yi = obs.current.yourIndex
    for entry in obs.logs:
        if entry.type == LogType.TURN_END:
            for prev in _cur_turn_logs:
                if prev.type == LogType.ATTACK and getattr(prev, 'playerIndex', yi) != yi:
                    _opp_last_attack_id = prev.attackId
            _cur_turn_logs.clear()
        else:
            _cur_turn_logs.append(entry)


# ── Board helpers ──

def read_deck_csv():
    fp = "deck.csv"
    if not os.path.exists(fp):
        fp = "/kaggle_simulations/agent/deck.csv"
    with open(fp) as f:
        return [int(line) for line in f.read().strip().split("\n")]


def get_card(obs, area, index, player_index):
    if area is None or index is None:
        return None
    ps = obs.current.players[player_index]
    if area == AreaType.DECK and obs.select and obs.select.deck is not None:
        return obs.select.deck[index] if index < len(obs.select.deck) else None
    if area == AreaType.HAND and ps.hand is not None:
        return ps.hand[index] if index < len(ps.hand) else None
    if area == AreaType.DISCARD:
        return ps.discard[index] if index < len(ps.discard) else None
    if area == AreaType.ACTIVE:
        return ps.active[index] if index < len(ps.active) else None
    if area == AreaType.BENCH:
        return ps.bench[index] if index < len(ps.bench) else None
    if area == AreaType.PRIZE:
        return ps.prize[index] if index < len(ps.prize) else None
    if area == AreaType.STADIUM:
        return obs.current.stadium[index] if index < len(obs.current.stadium) else None
    if area == AreaType.LOOKING and obs.current.looking is not None:
        return obs.current.looking[index] if index < len(obs.current.looking) else None
    return None


def option_card(obs, opt):
    yi = obs.current.yourIndex
    pi = opt.playerIndex if opt.playerIndex is not None else yi
    if opt.type == OptionType.PLAY:
        return get_card(obs, AreaType.HAND, opt.index, pi)
    return get_card(obs, opt.area, opt.index, pi)


def option_target(obs, opt):
    if opt.inPlayArea is None or opt.inPlayIndex is None:
        return None
    return get_card(obs, opt.inPlayArea, opt.inPlayIndex, obs.current.yourIndex)


def my_state(obs):
    return obs.current.players[obs.current.yourIndex]


def opp_state(obs):
    return obs.current.players[1 - obs.current.yourIndex]


def active_pokemon(obs):
    ps = my_state(obs)
    return ps.active[0] if ps.active else None


def opp_active_pokemon(obs):
    ps = opp_state(obs)
    return ps.active[0] if ps.active else None


def opp_bench_pokemon(obs):
    return [p for p in opp_state(obs).bench if p]


def all_my_pokemon(obs):
    ps = my_state(obs)
    return [p for p in (ps.active + ps.bench) if p]


def hand_ids(obs):
    hand = my_state(obs).hand
    return [c.id for c in hand if c] if hand else []


def discard_ids(obs):
    return [c.id for c in (my_state(obs).discard or []) if c]


def metal_in_discard(obs):
    return sum(1 for c in (my_state(obs).discard or []) if c and c.id == METAL_ENERGY)


def energy_count(pokemon):
    if pokemon is None:
        return 0
    if getattr(pokemon, "energyCards", None) is not None:
        return len(pokemon.energyCards)
    return len(getattr(pokemon, "energies", []) or [])


def retreat_cost(pokemon):
    data = CARD_DB.get(pokemon.id) if pokemon else None
    return getattr(data, "retreatCost", 0) if data else 0


def damage_on(pokemon):
    if pokemon is None:
        return 0
    return max(0, getattr(pokemon, "maxHp", pokemon.hp) - pokemon.hp)


def has_tool(pokemon):
    return bool(getattr(pokemon, "tools", []) or [])


def count_in_play(obs, card_id):
    return sum(1 for p in all_my_pokemon(obs) if p.id == card_id)


def has_in_play(obs, card_id):
    return any(p.id == card_id for p in all_my_pokemon(obs))


def need_duraludon(obs):
    return sum(1 for p in all_my_pokemon(obs) if p.id in {DURALUDON, ARCHALUDON_EX}) < 2


def need_archaludon(obs):
    has_dura, ex_count = False, 0
    for p in all_my_pokemon(obs):
        if p.id == DURALUDON:
            has_dura = True
        elif p.id == ARCHALUDON_EX:
            ex_count += 1
    return has_dura and ex_count < 2


def safe_discard_count(obs):
    ids = hand_ids(obs)
    mt = metal_in_discard(obs)
    safe = 0
    for cid in ids:
        if cid == METAL_ENERGY and mt + safe < 2:
            safe += 1
        elif cid == CINDERACE:
            safe += 1
    draw_in_hand = sum(1 for c in ids if c in (LILLIE, EXPLORER))
    if draw_in_hand >= 2:
        safe += draw_in_hand - 1
    return safe


def prize_value(pokemon):
    data = CARD_DB.get(pokemon.id) if pokemon else None
    if data and getattr(data, "megaEx", False):
        return 3
    if data and getattr(data, "ex", False):
        return 2
    return 1


def best_attack_damage(obs, attack_id):
    if attack_id == RAGING_HAMMER:
        return 80 + damage_on(active_pokemon(obs)) // 10 * 10
    return _ATTACK_BASE_DMG.get(attack_id, 0)


def is_metal_weak(pokemon):
    if pokemon is None:
        return False
    data = CARD_DB.get(pokemon.id)
    w = getattr(data, "weakness", None) if data else None
    if w is None:
        return False
    return getattr(w, "value", w) == METAL_ENERGY


def effective_damage(base_damage, target):
    return base_damage * 2 if is_metal_weak(target) else base_damage


def _first_option_index(obs, card_id):
    for o in obs.select.option:
        oc = option_card(obs, o)
        if oc and oc.id == card_id:
            return getattr(o, 'index', None)
    return None


# ── Attack routes ──

def direct_attack_energy_route(obs, pokemon):
    e = energy_count(pokemon)
    if e >= 3:
        return True, False
    if e == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):
        return True, True
    return False, False


def can_evolve_to_archaludon_now(pokemon, obs):
    if pokemon is None or pokemon.id != DURALUDON:
        return False
    if ARCHALUDON_EX not in hand_ids(obs):
        return False
    return not getattr(pokemon, "appearThisTurn", True)


def alloy_attack_energy_route(obs, pokemon):
    if not can_evolve_to_archaludon_now(pokemon, obs):
        return False, False
    current = energy_count(pokemon)
    alloy = min(2, metal_in_discard(obs))
    total = current + alloy
    if total >= 3:
        return True, False
    if total == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):
        return True, True
    return False, False


def attack_energy_route(obs, pokemon):
    if pokemon is None:
        return False, False
    if pokemon.id == ARCHALUDON_EX:
        return direct_attack_energy_route(obs, pokemon)
    if pokemon.id == DURALUDON:
        ok, uses_attach = direct_attack_energy_route(obs, pokemon)
        if ok:
            return True, uses_attach
        return alloy_attack_energy_route(obs, pokemon)
    return False, False


def archaludon_ex_attack_route(obs):
    active = active_pokemon(obs)
    if active and active.id in {ARCHALUDON_EX, DURALUDON}:
        ok, uses_attach = attack_energy_route(obs, active)
        if ok:
            return {"attacker": active, "uses_attach": uses_attach, "needs_retreat": False}

    if active is None or obs.current.retreated or energy_count(active) < retreat_cost(active):
        return None
    ps = my_state(obs)
    for pokemon in [p for p in ps.bench if p]:
        if pokemon.id not in {ARCHALUDON_EX, DURALUDON}:
            continue
        ok, uses_attach = attack_energy_route(obs, pokemon)
        if ok:
            return {"attacker": pokemon, "uses_attach": uses_attach, "needs_retreat": True}
    return None


def planned_archaludon_attacks(obs):
    route = archaludon_ex_attack_route(obs)
    if route is None:
        return []
    attacker = route["attacker"]
    attacks = []
    if attacker.id == ARCHALUDON_EX:
        attacks.append({"damage": 220})
        if has_in_play(obs, RELICANTH):
            attacks.append({"damage": 80 + damage_on(attacker) // 10 * 10})
    if attacker.id == DURALUDON:
        attacks.append({"damage": 80 + damage_on(attacker) // 10 * 10})
        if can_evolve_to_archaludon_now(attacker, obs):
            attacks.append({"damage": 220})
    return attacks


# ── Matchup detection & opponent max damage ──

ALAKAZAM_LINE = {741, 742, 743}
_ALA_BOARD_GAIN = {66: 3, 742: 2, 305: 2, 65: 2, 741: 1}  # Dudunsparce, Kadabra, Dunsparce×2, Abra


def _estimate_alakazam_from_pokes(opp, pokes):
    """(floor, ceiling, ceiling_with_boss) damage from visible Alakazam line."""
    ids = [p.id for p in pokes if p]
    if not (ALAKAZAM_LINE & set(ids)):
        return 0, 0, 0
    base = opp.handCount + 1
    gain = sum(_ALA_BOARD_GAIN.get(i, 0) for i in ids)
    enriching_seen = (
        any(c and c.id == 13 for c in (opp.discard or []))
        or any(c and c.id == 13 for p in pokes if p for c in (getattr(p, "energyCards", None) or []))
    )
    if not enriching_seen:
        gain += 3
    if any(i == 140 for i in ids):
        gain += 3
    return base * 20, (base + gain + 2) * 20, (base + gain - 1) * 20


def _estimate_alakazam(obs):
    """(floor, ceiling, ceiling_with_boss) damage from Powerful Hand."""
    opp = opp_state(obs)
    pokes = ([opp.active[0]] if opp.active else []) + list(opp.bench or [])
    return _estimate_alakazam_from_pokes(opp, pokes)


def detect_matchup(obs):
    opp = opp_state(obs)
    ids = {p.id for p in (opp.active + opp.bench) if p}
    if ids & CRUSTLE_LINE:
        return "crustle"
    if ids & HOP_LINE:
        return "hop"
    if ids & STARMIE_LINE:
        return "starmie"
    if ids & LUCARIO_LINE:
        return "lucario"
    if ids & ALAKAZAM_LINE:
        return "alakazam"
    return "generic"


def opp_max_damage(obs):
    matchup = detect_matchup(obs)
    if matchup == "alakazam":
        _, ceiling, _ = _estimate_alakazam(obs)
        return ceiling
    if matchup == "crustle":
        return 120
    if matchup == "hop":
        return 220
    if matchup == "lucario":
        return 270  # Mega Brave base. PPP adds +30 each but unpredictable
    if matchup == "starmie":
        return 210
    return 220


# ── Overrides ──

def apply_overrides(obs, opt, score, reason):
    # Hard rule: don't Explorer with low deck
    if opt.type == OptionType.PLAY:
        card = option_card(obs, opt)
        cid = card.id if card else None
        if my_state(obs).deckCount <= 10 and cid == EXPLORER:
            return -5000, "hard: don't Explorer with low deck"

    if detect_matchup(obs) != "crustle":
        return score, reason

    # Crustle overrides
    card = option_card(obs, opt)
    cid = card.id if card else getattr(opt, 'cardId', None)
    ctx = obs.select.context

    if opt.type == OptionType.EVOLVE and cid == ARCHALUDON_EX:
        return -10000, "Crustle: don't evolve to ex"

    if opt.type == OptionType.ATTACK:
        aid = getattr(opt, 'attackId', None)
        active = active_pokemon(obs)
        opp_act = opp_active_pokemon(obs)
        opp_has_spiky = bool(opp_act and any(
            getattr(c, 'id', None) == 14
            for c in (getattr(opp_act, 'energyCards', None) or [])))
        if (active and active.id == DURALUDON and active.hp == 130
                and opp_act and opp_act.id == 345 and energy_count(opp_act) >= 2
                and opp_has_spiky):
            return -3000, "Crustle: full HP Duraludon waits out Spiky"
        if aid == METAL_DEFENDER:
            return -5000, "Crustle: Metal Defender does 0"
        if aid == RAGING_HAMMER:
            rh_dmg = 80 + damage_on(active_pokemon(obs)) // 10 * 10
            return max(score, 200), "Crustle: Raging Hammer"

    if opt.type == OptionType.PLAY:
        if cid == RELICANTH:
            return -5000, "Crustle: skip Relicanth"
        dc = my_state(obs).deckCount
        if dc <= 10 and cid in (EXPLORER, LILLIE):
            if cid == LILLIE and dc <= 3 and my_state(obs).handCount >= dc + 6:
                return 15000, "Crustle: Lillie to refill deck"
            return -5000, "Crustle: don't draw with low deck"
        if cid == LILLIE:
            has_metal = any(c and c.id == METAL_ENERGY for c in (my_state(obs).hand or []) if c)
            if not has_metal:
                return score, "Crustle: Lillie OK (no energy in hand)"

    if opt.type == OptionType.ATTACH:
        target = option_target(obs, opt)
        tid = target.id if target else None
        if getattr(opt, 'inPlayArea', None) == AreaType.BENCH and tid == DURALUDON:
            return score + 10000, "Crustle: bench Duraludon energy priority"
        if getattr(opt, 'inPlayArea', None) == AreaType.ACTIVE:
            active = active_pokemon(obs)
            if active and energy_count(active) >= 2:
                return score + 3000, "Crustle: Active 3rd energy"

    if ctx == SelectContext.TO_HAND and opt.type == OptionType.CARD and cid == ARCHALUDON_EX:
        return -3000, "Crustle: skip Archaludon ex"

    if ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:
        if cid == ARCHALUDON_EX and score < 0:
            return 9000, "Crustle: discard Archaludon ex"

    return score, reason


# ── Scoring ──

def score_setup(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else None
    ctx = obs.select.context

    if ctx == SelectContext.MULLIGAN:
        return (10000, "no mulligan") if opt.type == OptionType.NO else (0, "mulligan")
    if ctx == SelectContext.IS_FIRST:
        return (10000, "choose second") if opt.type == OptionType.NO else (0, "go first")
    if ctx == SelectContext.SETUP_ACTIVE_POKEMON:
        return _SETUP_ACTIVE_PRIORITY.get(cid, (0, "unknown Active"))
    if ctx == SelectContext.SETUP_BENCH_POKEMON:
        return -10000, "never bench during setup"
    return 0, "non-setup"


# HP threshold per matchup: skip Ice Cream if HP > this value
_ICE_CREAM_HP_THRESHOLD = {
    "lucario": 270,
    "starmie": 210,
    "crustle": 120,
    "hop": 220,
    "generic": 230,
}


def should_skip_ice_cream(obs, active):
    """Decide whether to skip Jumbo Ice Cream. Returns (skip: bool, reason: str)."""
    # 1. Active must be Archaludon ex
    if active.id != ARCHALUDON_EX:
        return True, "skip Ice Cream: not Archaludon ex"
    # 2. Raging Hammer KO guard: don't heal if it loses a KO (but 220 Metal Defender still KOs → heal OK)
    opp_act = opp_active_pokemon(obs)
    if opp_act and has_in_play(obs, RELICANTH):
        md_kills = effective_damage(220, opp_act) >= opp_act.hp
        if not md_kills:
            rh_dmg = 80 + damage_on(active) // 10 * 10
            rh_after = 80 + max(0, damage_on(active) - 80) // 10 * 10
            if effective_damage(rh_dmg, opp_act) >= opp_act.hp and effective_damage(rh_after, opp_act) < opp_act.hp:
                return True, "skip Ice Cream: healing loses Raging Hammer KO"
    # 3. Alakazam: all-or-nothing Ice Cream decision
    matchup = detect_matchup(obs)
    if matchup == "alakazam":
        floor, ceiling, _ = _estimate_alakazam(obs)
        opp_a = opp_active_pokemon(obs)
        attacks = planned_archaludon_attacks(obs)
        if opp_a and attacks and any(effective_damage(a["damage"], opp_a) >= opp_a.hp for a in attacks):
            _, ceiling, _ = _estimate_alakazam_from_pokes(opp_state(obs), opp_bench_pokemon(obs))
        ice_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id == JUMBO_ICE_CREAM)
        max_hp = getattr(active, "maxHp", active.hp)
        hp_after_all = min(max_hp, active.hp + ice_count * 80)
        if hp_after_all <= active.hp:
            return True, "skip Ice Cream: no effective healing"
        if hp_after_all < floor:
            return True, f"skip Ice Cream: even {ice_count}x heal ({hp_after_all}) < floor {floor}"
        if hp_after_all >= ceiling:
            return False, f"use Ice Cream: {ice_count}x heal ({hp_after_all}) >= ceil {ceiling}"
        return False, f"use Ice Cream: {ice_count}x heal ({hp_after_all}) between floor={floor} ceil={ceiling}"
    # 4. HP above matchup threshold
    threshold = _ICE_CREAM_HP_THRESHOLD.get(matchup, 220)
    if active.hp > threshold:
        return True, f"skip Ice Cream: HP {active.hp} > {threshold} ({matchup})"
    # 5. Use it
    return False, ""


ITEMS = {POKE_PAD, ULTRA_BALL, POKEGEAR, NIGHT_STRETCHER, JUMBO_ICE_CREAM, HERO_CAPE}


def score_play(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else None
    ids = hand_ids(obs)

    # ── Pokemon: bench if available ──
    if cid in {DURALUDON, RELICANTH}:
        return 18000, "play Pokemon"

    # ── Stadium ──
    if cid == FULL_METAL_LAB:
        active = active_pokemon(obs)
        if active and active.id not in {DURALUDON, ARCHALUDON_EX}:
            return -200, "skip FML: Active not Metal"
        return 20000, "play Full Metal Lab"

    # ── Items: default 20000, only negative exceptions ──
    if cid in ITEMS:
        if cid == HERO_CAPE:
            if not any(p.id in {ARCHALUDON_EX, DURALUDON} and not has_tool(p) for p in all_my_pokemon(obs)):
                return -500, "save Hero's Cape: no target"
        if cid == JUMBO_ICE_CREAM:
            active = active_pokemon(obs)
            if active:
                skip, reason = should_skip_ice_cream(obs, active)
                if skip:
                    return -500, reason
        if cid == NIGHT_STRETCHER:
            disc = discard_ids(obs)
            has_urgent = (
                (DURALUDON in disc and DURALUDON not in ids and count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX) <= 1)
                or (ARCHALUDON_EX in disc and ARCHALUDON_EX not in ids and has_in_play(obs, DURALUDON))
                or (METAL_ENERGY in disc and not obs.current.energyAttached
                    and sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY) == 0
                    and any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) == 2 for p in all_my_pokemon(obs)))
            )
            if not has_urgent:
                return -500, "save Night Stretcher"
        if cid == ULTRA_BALL:
            bench_empty = len([p for p in my_state(obs).bench if p]) == 0
            if bench_empty:
                return 300, "Ultra Ball: bench empty (donk risk)"
            metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)
            metal_in_trash = metal_in_discard(obs)
            if metal_in_trash == 0 and metal_in_hand >= 1:
                return 20000, "Ultra Ball: fuel Alloy"
            if safe_discard_count(obs) >= 2 and (need_archaludon(obs) or need_duraludon(obs)):
                return 20000, "Ultra Ball: search line"
            return -1000, "skip Ultra Ball"
        return 20000, "play item"

    if cid == EXPLORER:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        return 16000, "play Explorer"

    if cid == LILLIE:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        if BOSS in ids and planned_archaludon_attacks(obs):
            return -500, "save Lillie: Boss in hand with attacker ready"
        return 5000, "play Lillie"

    if cid == BOSS:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        # vs Hop: Boss Snorlax to remove Extra Helpings (+30) ASAP
        if detect_matchup(obs) == "hop":
            active = active_pokemon(obs)
            opp_has_snorlax = any(p.id == HOP_SNORLAX for p in opp_bench_pokemon(obs))
            if opp_has_snorlax and active:
                # Case 1: Cinderace active + bench has Duraludon → Turbo Flare Snorlax
                if active.id == CINDERACE:
                    has_dura_bench = any(p.id in {DURALUDON, ARCHALUDON_EX}
                                        for p in my_state(obs).bench if p)
                    if has_dura_bench:
                        return 16500, "Boss: pull Snorlax (Cinderace Turbo Flare)"
                # Case 2: Archaludon active, HP > 220, can attack → Boss Snorlax
                if active.id == ARCHALUDON_EX and active.hp > 220:
                    ok, _ = attack_energy_route(obs, active)
                    if ok:
                        return 16500, "Boss: pull Snorlax (Arch can tank Revenge 220)"
        if _opp_last_attack_id == MEGA_BRAVE:
            return -500, "save Boss: Mega Brave stuck"
        attacks = planned_archaludon_attacks(obs)
        if not attacks:
            return -500, "save Boss: no attacker"
        opp_act = opp_active_pokemon(obs)
        can_ko_active = opp_act and any(
            effective_damage(atk["damage"], opp_act) >= opp_act.hp for atk in attacks)
        remaining = len(my_state(obs).prize)
        if can_ko_active:
            if prize_value(opp_act) >= remaining:
                return -500, "save Boss: Active KO wins"
            for target in opp_bench_pokemon(obs):
                for atk in attacks:
                    if effective_damage(atk["damage"], target) >= target.hp:
                        if prize_value(target) >= remaining:
                            return 20000, "LETHAL Boss"
                        break
            return -500, "save Boss: can KO Active"
        best_score = -500
        best_reason = "save Boss"
        for target in opp_bench_pokemon(obs):
            for atk in attacks:
                if effective_damage(atk["damage"], target) >= target.hp:
                    pv = prize_value(target)
                    if pv >= remaining:
                        return 20000, "LETHAL Boss"
                    s = 4000 + pv * 200 + energy_count(target) * 100
                    if s > best_score:
                        best_score = s
                        best_reason = "Boss: pull bench target"
                    break
        if best_score <= 0:
            metal_total = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)
            metal_total += sum(energy_count(p) for p in all_my_pokemon(obs) if p)
            has_cind = has_in_play(obs, CINDERACE)
            draw_in_hand = any(c and c.id in (EXPLORER, LILLIE) for c in (my_state(obs).hand or []) if c)
            if metal_total <= 2 and not has_cind and not draw_in_hand:
                best_stall = -500
                stall_reason = "save Boss"
                for target in opp_bench_pokemon(obs):
                    te = energy_count(target)
                    cd = CARD_DB.get(target.id)
                    rc = cd.retreatCost if cd else 0
                    min_atk = 99
                    if cd and cd.attacks:
                        for aid in cd.attacks:
                            atk = ALL_ATTACKS.get(aid)
                            if atk:
                                min_atk = min(min_atk, len(atk.energies))
                    if min_atk == 99:
                        min_atk = 1
                    ss = 4000 + rc * 1000 + min_atk * 500 - te * 800
                    if ss > best_stall:
                        best_stall = ss
                        stall_reason = "Boss stall"
                return best_stall, stall_reason
        return best_score, best_reason

    return 1000, "generic play"


def score_evolve(obs, opt):
    card = option_card(obs, opt)
    target = option_target(obs, opt)
    cid = card.id if card else None
    tid = target.id if target else None
    if cid == ARCHALUDON_EX and tid == DURALUDON:
        target_is_active = opt.inPlayArea == AreaType.ACTIVE
        mc = metal_in_discard(obs)
        if target_is_active:
            if energy_count(target) >= 3 and not has_in_play(obs, ARCHALUDON_EX):
                return 17000, "evolve Active 3-energy Duraludon"
            if mc >= 2:
                return 28000 + mc * 2000, "evolve Active Duraludon"
            if mc == 1:
                return 8000, "delay Active evolve: 1 Metal"
            return -500, "hold: no Metal in discard"
        if mc >= 2:
            return 14000 + mc * 1000, "evolve Bench Duraludon"
        return -1000, "hold: evolve Active first"
    return 10000, "generic evolution"


def attach_target_score(obs, target, area):
    if target is None:
        return 0
    cid = target.id
    e = energy_count(target)

    if e >= 3:
        return -5000
    if cid == CINDERACE and e >= 1:
        return -3000

    score = 0
    if cid == CINDERACE:
        score = 3000
        if e == 0:
            score += 7000 + (12000 if area == AreaType.ACTIVE else 5000)
    elif cid in {DURALUDON, ARCHALUDON_EX}:
        score = 6000 if cid == ARCHALUDON_EX else 5500
        score += {2: 12000, 1: 7000, 0: 4000}.get(e, -1000)
        score += 1000 if area == AreaType.ACTIVE else 500
    else:
        score = 1000 + (1000 if e == 0 else 0)

    # HP-based adjustment
    if target.hp > 0:
        max_hp = getattr(target, "maxHp", target.hp)
        ratio = target.hp / max_hp if max_hp > 0 else 1
        if ratio <= 0.25:
            score -= 1500
        elif ratio <= 0.50:
            score -= 500
        else:
            score += min(1000, target.hp // 40 * 100)
    return score


def score_attach(obs, opt):
    card = option_card(obs, opt)
    target = option_target(obs, opt)
    cid = card.id if card else None
    tid = target.id if target else None

    if cid == HERO_CAPE:
        if tid == ARCHALUDON_EX and target and not has_tool(target):
            return 11000, "Hero's Cape on Archaludon ex"
        if tid == DURALUDON and target and not has_tool(target) and energy_count(target) >= 1:
            return 8000, "Hero's Cape on Duraludon"
        return -1000, "save Hero's Cape"

    if cid != METAL_ENERGY:
        return -500, "skip non-Metal"
    if obs.current.energyAttached:
        return -1000, "already attached"

    return attach_target_score(obs, target, opt.inPlayArea), "attach Metal"


def score_retreat(obs, opt):
    active = active_pokemon(obs)
    if active and active.id == ARCHALUDON_EX and has_tool(active) and active.hp > 200:
        return -5000, "don't retreat HP400 tank"
    route = archaludon_ex_attack_route(obs)
    if route and route["needs_retreat"]:
        return 13000, "retreat to attack-ready ex"
    return -100, "avoid retreat"


_MAIN_DISPATCH = {
    OptionType.PLAY: score_play, OptionType.EVOLVE: score_evolve,
    OptionType.ATTACH: score_attach, OptionType.RETREAT: score_retreat,
}


def score_option(obs, opt):
    ctx = obs.select.context

    if ctx in {SelectContext.IS_FIRST, SelectContext.MULLIGAN,
               SelectContext.SETUP_ACTIVE_POKEMON, SelectContext.SETUP_BENCH_POKEMON}:
        return score_setup(obs, opt)

    if opt.type in {OptionType.YES, OptionType.NO}:
        if ctx == SelectContext.IS_FIRST:
            return score_setup(obs, opt)
        if ctx == SelectContext.ACTIVATE:
            return (100000, "Explosiveness") if opt.type == OptionType.YES else (-100000, "never decline")
        return (1, "yes") if opt.type == OptionType.YES else (0, "no")

    if opt.type == OptionType.NUMBER:
        return (opt.number or 0), "number"

    if ctx == SelectContext.MAIN:
        fn = _MAIN_DISPATCH.get(opt.type)
        if fn:
            score, reason = fn(obs, opt)
        elif opt.type == OptionType.ABILITY:
            score, reason = 1, "ability"
        elif opt.type == OptionType.ATTACK:
            score, reason = best_attack_damage(obs, opt.attackId), "attack"
        elif opt.type == OptionType.END:
            score, reason = 0, "end turn"
        else:
            score, reason = 500, "generic MAIN"
    elif ctx == SelectContext.TO_HAND:
        score, reason = score_to_hand(obs, opt)
    elif ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:
        score, reason = score_discard(obs, opt)
    elif ctx in {SelectContext.ATTACH_TO, SelectContext.TO_FIELD, SelectContext.TO_BENCH,
                 SelectContext.ATTACH_FROM, SelectContext.SWITCH, SelectContext.TO_ACTIVE,
                 SelectContext.HEAL, SelectContext.DAMAGE}:
        score, reason = score_target(obs, opt)
    elif ctx == SelectContext.ATTACK:
        score, reason = best_attack_damage(obs, opt.attackId), "attack"
    elif opt.type == OptionType.CARD:
        score, reason = score_to_hand(obs, opt)
    elif opt.type == OptionType.ENERGY:
        score, reason = 1000, "energy"
    elif opt.type == OptionType.END:
        score, reason = 0, "end"
    else:
        score, reason = 100, "fallback"

    return apply_overrides(obs, opt, score, reason)


def score_to_hand(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ids = hand_ids(obs)
    effect = getattr(obs.select, "effect", None)
    effect_id = effect.id if effect else None

    if effect_id == EXPLORER:
        has_ready = any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) >= 3
                        for p in all_my_pokemon(obs))
        metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)

        if cid == HERO_CAPE:
            has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))
            return (27000 if has_target else 22000), "Explorer: Hero's Cape"
        if cid == METAL_ENERGY:
            if has_ready or metal_in_hand > 0:
                return 0, "Explorer: skip energy"
            if getattr(opt, 'index', 0) == _first_option_index(obs, METAL_ENERGY):
                return 25000, "Explorer: take 1st energy"
            return 0, "Explorer: skip 2nd energy"
        if cid == ARCHALUDON_EX and need_archaludon(obs):
            return 20000, "Explorer: take Archaludon ex"
        if cid == DURALUDON and need_duraludon(obs):
            return 18000, "Explorer: take Duraludon"
        if cid == RELICANTH and not has_in_play(obs, RELICANTH) and RELICANTH not in ids:
            return 15000, "Explorer: take Relicanth"
        sup_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id in (EXPLORER, LILLIE))
        if cid in (EXPLORER, LILLIE) and sup_count == 0:
            return 12000, "Explorer: take supporter"
        return 0, "Explorer: let discard"

    dura_ex_count = count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX)
    if cid == DURALUDON and DURALUDON not in ids and dura_ex_count <= 1:
        return 22000, "take Duraludon: backup"
    if cid == ARCHALUDON_EX and need_archaludon(obs):
        return 20000, "take Archaludon ex"
    if cid == DURALUDON and need_duraludon(obs):
        return 18000, "take Duraludon"
    if cid == CINDERACE:
        return -2000, "skip Cinderace"
    if cid == RELICANTH and not has_in_play(obs, RELICANTH):
        return 9000, "take Relicanth"
    if cid == METAL_ENERGY:
        return 8000, "take Metal Energy"
    if cid == EXPLORER and not obs.current.supporterPlayed:
        return 7500, "take Explorer"
    if cid == LILLIE and not obs.current.supporterPlayed:
        return 6500, "take Lillie"
    if cid == HERO_CAPE:
        has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))
        return (6000, "take Hero's Cape") if has_target else (1000, "generic take")
    if cid == FULL_METAL_LAB:
        return 5000, "take Full Metal Lab"
    if cid == BOSS:
        return 2500, "take Boss"
    return 1000, "generic take"


def score_discard(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ids = hand_ids(obs)
    mt = metal_in_discard(obs)
    effect = getattr(obs.select, "effect", None)
    effect_id = effect.id if effect else None

    if effect_id == ULTRA_BALL:
        mh = ids.count(METAL_ENERGY)
        if cid == METAL_ENERGY:
            if mt < 2 and mh >= 1:
                if getattr(opt, 'index', None) == _first_option_index(obs, METAL_ENERGY):
                    return 20000, "UB: 1st Metal"
                return 8000, "UB: 2nd Metal"
            return 8000, "UB: Metal"
        if cid == CINDERACE:
            return (18000, "UB: Cinderace") if (mt >= 2 or mh == 0) else (14000, "UB: Cinderace")
        draw_count = ids.count(LILLIE) + ids.count(EXPLORER)
        if cid in (LILLIE, EXPLORER) and draw_count >= 2:
            return (12000 if cid == LILLIE else 11000), "UB: surplus supporter"
        if cid == ULTRA_BALL and ids.count(ULTRA_BALL) > 1:
            return 10000, "UB: duplicate"
        if cid in (LILLIE, EXPLORER) and draw_count <= 1:
            return -3000, "UB: keep last supporter"

    if cid == METAL_ENERGY:
        if mt < 2:
            return 15000, "discard Metal"
        return (12000, "discard extra Metal") if ids.count(METAL_ENERGY) > 1 else (-1000, "keep last Metal")
    if cid == CINDERACE:
        return 10000, "discard Cinderace"
    if cid in {BOSS, FULL_METAL_LAB, POKEGEAR}:
        return 8500, "discard utility"
    if cid in {LILLIE, EXPLORER} and ids.count(cid) > 1:
        return 8000, "discard duplicate supporter"
    if cid == RELICANTH and (has_in_play(obs, RELICANTH) or ids.count(RELICANTH) > 1):
        return 6500, "discard extra Relicanth"
    if cid == ARCHALUDON_EX:
        return -5000, "keep Archaludon ex"
    if cid == DURALUDON:
        return -4000, "keep Duraludon"
    return 1000, "generic discard"


def score_target(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ctx = obs.select.context

    if ctx == SelectContext.ATTACH_TO:
        return (5000, "Metal") if cid == METAL_ENERGY else (1000, "attach")

    if ctx == SelectContext.ATTACH_FROM:
        if card and energy_count(card) >= 3:
            return -5000, "skip: 3+ energy"
        if card and cid == CINDERACE and energy_count(card) >= 1:
            return -3000, "skip: Cinderace ready"
        return attach_target_score(obs, card, opt.area), "effect attach"

    if ctx in {SelectContext.TO_FIELD, SelectContext.TO_BENCH}:
        if cid == ARCHALUDON_EX:
            return 18000, "target Archaludon ex"
        if cid == DURALUDON:
            return 16000, "target Duraludon"
        if cid == CINDERACE:
            return 3000, "avoid Cinderace"

    if ctx == SelectContext.HEAL:
        return (20000 + damage_on(card), "heal Archaludon ex") if cid == ARCHALUDON_EX else (damage_on(card), "heal")

    if ctx in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:
        yi = obs.current.yourIndex
        pi = getattr(opt, 'playerIndex', yi)
        if pi != yi and card:
            # vs Hop: prioritize Snorlax (remove Extra Helpings)
            if detect_matchup(obs) == "hop" and cid == HOP_SNORLAX and card:
                active = active_pokemon(obs)
                e = energy_count(card)
                tools = len(getattr(card, 'tools', None) or [])
                if active and active.id == CINDERACE:
                    # Cinderace: pull the least mobile Snorlax (low energy, no tools, high HP)
                    return 30000 - e * 100 - tools * 50 + card.hp, "Boss: Snorlax (immobile target)"
                else:
                    # Archaludon: pull the most threatening Snorlax (high energy, tools, high HP)
                    return 30000 + e * 100 + tools * 50 + card.hp, "Boss: Snorlax (biggest threat)"
            pv = prize_value(card)
            te = energy_count(card)
            killable = any(effective_damage(a["damage"], card) >= card.hp
                           for a in planned_archaludon_attacks(obs))
            if killable:
                return 20000 + pv * 3000 + te * 100, "Boss: KO"
            return 5000 + pv * 1000 + te * 200, "Boss: drag"
        if cid == CINDERACE:
            return 16000, "promote Cinderace (retreat 0)"
        if cid == ARCHALUDON_EX:
            return 15000, "promote Archaludon ex"
        if cid == DURALUDON:
            return 8000, "promote Duraludon"
        return 1000, "generic promote"

    if ctx == SelectContext.DAMAGE:
        hp = getattr(card, "hp", 999) if card else 999
        return 10000 - hp, "damage: lowest HP"

    return 1000, "generic target"


# ── Choose & Agent ──

def choose_options(obs):
    scored = []
    for i, opt in enumerate(obs.select.option):
        try:
            score, reason = score_option(obs, opt)
        except Exception as e:
            score, reason = -999999, f"error {type(e).__name__}: {e}"
        scored.append((score, i, reason))

    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)

    selected = []
    for score, i, reason in scored:
        if len(selected) >= obs.select.maxCount:
            break
        if score < 0 and len(selected) >= obs.select.minCount:
            continue
        selected.append(i)

    if len(selected) < obs.select.minCount:
        selected = [i for _, i, _ in scored[:obs.select.minCount]]

    return selected


def agent(obs_dict):
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        global _opp_last_attack_id, _cur_turn_logs
        _opp_last_attack_id = None
        _cur_turn_logs.clear()
        return my_deck
    _update_opp_attack_tracking(obs)
    if not obs.select.option:
        return []
    try:
        return choose_options(obs)
    except Exception:
        return random.sample(list(range(len(obs.select.option))), obs.select.maxCount)

In [ ]:
import opp_abomasnow
import opp_iono
import opp_lucario
import opp_archaludon

OPPONENTS = {
    "mirror_dragapult" : (EXPERT_DRAGAPULT_AGENT, DRAGAPULT_DECK),
    "abomasnow": (opp_abomasnow.agent, opp_abomasnow.my_deck),
    "iono" : (opp_iono.agent, opp_iono.my_deck),
    "lucario" : (opp_lucario.agent, opp_lucario.my_deck),
    "archaludon" : (opp_archaludon.agent, opp_archaludon.my_deck),
}

## Featurizers
Same featurizers as BC — must match exactly for weight compatibility.


In [ ]:
def safe_int(val, default=0):
    try: return int(val)
    except: return default

def enum_int(x, default=0):
    """Turn an int OR an enum into a plain int."""
    try: return int(x)
    except (TypeError, ValueError):
        try: return int(x.value)
        except Exception: return default

# ── Vocabulary of the cards the model should recognize (your deck's IDs) ─────
CARD_VOCAB = [119,120,121,140,184,235,1071,1079,1080,1086,1097,1120,1121,1152,
              1156,1182,1198,1210,1227,1256,2,5]
CARD_TO_IDX = {c: i for i, c in enumerate(CARD_VOCAB)}
VOCAB_SIZE  = len(CARD_VOCAB) + 1     # +1 = "other/unknown" (e.g. opponent cards)

# ── Option-type vocabulary (built from the real enum members) ────────────────
OPTION_TYPES = [OptionType.NUMBER, OptionType.YES, OptionType.CARD, OptionType.PLAY,
                OptionType.ATTACH, OptionType.EVOLVE, OptionType.ABILITY,
                OptionType.RETREAT, OptionType.ATTACK, OptionType.ENERGY, OptionType.ENERGY_CARD]
OPTION_TYPE_IDX = {enum_int(t): i for i, t in enumerate(OPTION_TYPES)}
N_TYPES = len(OPTION_TYPES)

def _count_vec(cards):
    vec = [0.0] * VOCAB_SIZE
    for c in (cards or []):
        if not isinstance(c, dict):
            continue
        vec[CARD_TO_IDX.get(c.get("id"), VOCAB_SIZE - 1)] += 1.0
    return vec

def _count_vec(cards):
    vec = [0.0] * VOCAB_SIZE
    for c in (cards or []):
        if not isinstance(c, dict):
            continue
        vec[CARD_TO_IDX.get(c.get("id"), VOCAB_SIZE - 1)] += 1.0
    return vec

def featurize_state(obs_dict):
    cur = obs_dict.get("current") or {}
    players = cur.get("players") or [{}, {}]
    my_idx = safe_int(cur.get("yourIndex", 0))
    op_idx = 1 - my_idx
    me = players[my_idx] if len(players) > my_idx else {}
    op = players[op_idx] if len(players) > op_idx else {}

    def pf(p):
        p = p or {}
        active = p.get("active") or []
        fa = active[0] if (active and active[0] is not None) else {}
        return [safe_int(p.get("handCount", 0)), safe_int(p.get("deckCount", 0)),
                len(p.get("prize") or []), len(p.get("bench") or []),
                safe_int(fa.get("hp", 0)), len(fa.get("energies") or [])]

    mf, of = pf(me), pf(op)
    sel = obs_dict.get("select") or {}
    base = mf + of + [
        mf[2] - of[2],
        1 if cur.get("supporterPlayed") else 0,
        safe_int(cur.get("turn", 0)),
        1 if cur.get("stadium") else 0,
        safe_int(sel.get("context", 0)),
        safe_int(sel.get("minCount", 1)),
        safe_int(sel.get("maxCount", 1)),
    ]

    # NEW: board composition over the card vocab (what the heuristic decides on)
    me = me or {}
    my_field = (me.get("active") or []) + (me.get("bench") or [])
    field_vec   = _count_vec(my_field)
    hand_vec    = _count_vec(me.get("hand"))
    discard_vec = _count_vec(me.get("discard"))

    return np.array(base + field_vec + hand_vec + discard_vec, dtype=np.float32)

def _resolve_card(obs, o, my_idx):
    """Best-effort: which Card/Pokemon does this option concern? (reuses get_card)"""
    try:
        t = o.type
        if t == OptionType.PLAY:
            return get_card(obs, AreaType.HAND, o.index, my_idx)
        if t == OptionType.ATTACH or t == OptionType.EVOLVE:
            return get_card(obs, o.inPlayArea, o.inPlayIndex, my_idx)
        if t == OptionType.CARD or t == OptionType.ABILITY or t == OptionType.ENERGY or t == OptionType.ENERGY_CARD:
            return get_card(obs, o.area, o.index, getattr(o, "playerIndex", my_idx))
    except Exception:
        return None
    return None

def featurize_option(obs, o, my_idx):
    """Now includes WHICH CARD the option is about — the key fix."""
    # 1) option type (one-hot)
    tvec = [0.0] * N_TYPES
    tvec[OPTION_TYPE_IDX.get(enum_int(getattr(o, "type", 0)), N_TYPES - 1)] = 1.0

    # 2) card identity (one-hot over the deck vocab; unknown -> last slot)
    cvec = [0.0] * VOCAB_SIZE
    is_pkmn, hp, ecount = 0.0, 0.0, 0.0
    card = _resolve_card(obs, o, my_idx)
    if card is not None:
        cvec[CARD_TO_IDX.get(getattr(card, "id", None), VOCAB_SIZE - 1)] = 1.0
        if isinstance(card, Pokemon):
            is_pkmn = 1.0
            hp = float(getattr(card, "hp", 0) or 0)
            ecount = float(len(getattr(card, "energies", []) or []))
    else:
        cvec[VOCAB_SIZE - 1] = 1.0

    # 3) properties (normalized so magnitudes stay reasonable)
    attack_id = float(enum_int(getattr(o, "attackId", 0)))
    return np.array(tvec + cvec + [is_pkmn, hp / 300.0, ecount, attack_id / 1000.0],
                    dtype=np.float32)

STATE_DIM  = 19 + 3 * VOCAB_SIZE      # 19 + 69 = 88
OPTION_DIM = N_TYPES + VOCAB_SIZE + 4 # 38
INPUT_DIM  = STATE_DIM + OPTION_DIM   # 126
print("INPUT_DIM =", INPUT_DIM)

## BC Model Architecture (for loading weights)


In [ ]:
class ImitationNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
    def forward(self,x):
        return self.net(x).squeeze(-1)

## Load BC Model

The BC training notebook saves `model.pt` to its output. Add that notebook's
output as a data source, then load it here. Update the path below to match
your notebook slug.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD THE BC MODEL — update the path to match your BC notebook's output slug
# ═══════════════════════════════════════════════════════════════════════════
BC_MODEL_PATH = "/kaggle/input/bc-training/model.pt"  # ← update this slug

# Fallback paths (try multiple locations)
CANDIDATE_PATHS = [
    BC_MODEL_PATH,
    "/kaggle/input/bc-training/model_dagger_final.pt",
    "model.pt",  # if running locally
]

bc_model = ImitationNet(INPUT_DIM)
loaded = False
for path in CANDIDATE_PATHS:
    if os.path.exists(path):
        bc_model.load_state_dict(torch.load(path, map_location="cpu"))
        bc_model.eval()
        loaded = True
        print(f"✓ Loaded BC model from: {path}")
        break

if not loaded:
    print("✗ ERROR: Could not find model.pt!")
    print("  Make sure you added the BC notebook output as a data source.")
    print("  Expected at:", BC_MODEL_PATH)

# Reinforcement Learning — Prep

Everything below turns the imitation setup into an **RL-ready** foundation. Run
the cells in order; nothing here changes your BC/DAgger cells above.

**What each section gives you**
1. `PokemonRLEnv` — a Gym-style `reset()`/`step()` wrapper around the engine. It hides opponent turns and non-decision prompts, and hands back a reward each step.
2. `ActorCritic` — a policy (softmax over the legal options) **plus** a value head, warm-startable from your BC weights via `warm_start_from_bc`.
3. Helpers to turn a timestep into network inputs and to sample a (possibly multi-pick) action.
4. `run_episode` — plays one game and records the transitions PPO needs.
5. `compute_gae` + `ppo_update` — advantages and one clipped PPO step.
6. `train_rl` — a minimal end-to-end PPO loop against your opponent pool.
7. `rl_smoke_test` — **run this first** to confirm the wiring before real training.

> Prereqs already defined above: `battle_start/select/finish`, `to_observation_class`,
> `featurize_state`, `featurize_option`, `safe_int`, `DRAGAPULT_DECK`, `OPPONENTS`,
> `ImitationNet`, `STATE_DIM/OPTION_DIM/INPUT_DIM`, and the torch/np imports.


### 1. Environment wrapper
Our agent is player 0. Between our decisions the env auto-plays the opponent and auto-answers non-decision prompts (e.g. deck return when `select is None`). Reward = prize-swing shaping each step, plus a terminal ±1 for the win/loss.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 1 — Environment wrapper (the single most important piece)
# ---------------------------------------------------------------------------
class PokemonRLEnv:
    """A Gym-style wrapper around the cg battle engine.

    Our learning agent is ALWAYS player 0; the opponent is a fixed scripted
    agent (player 1). One env instance == one opponent matchup.

    The engine is turn-interleaved and also asks *us* for non-decisions
    (e.g. returning the deck at setup when select is None). This wrapper hides
    all of that: between two of OUR real decisions it auto-plays the opponent
    and auto-answers our non-decision prompts, so from the agent's point of
    view it looks like a clean reset()/step() loop.

    A "timestep" dict returned by reset()/step() contains:
        obs_dict : raw engine dict
        obs      : parsed Observation (only when it's our decision)
        options  : list of legal options at this decision (empty if done)
        my_idx   : our player index (0)
        reward   : reward earned since the previous decision
        done     : True if the episode is over
    """

    def __init__(self, opp_agent, opp_deck, my_deck=None,
                 reward_shaping=True, shape_scale=0.2, max_advance=10000):
        self.opp_agent = opp_agent
        self.opp_deck = opp_deck
        self.my_deck = my_deck if my_deck is not None else DRAGAPULT_DECK
        self.reward_shaping = reward_shaping
        self.shape_scale = shape_scale
        self.max_advance = max_advance
        self.obs_dict = None
        self.done = True
        self._prev_prizes = None

    # -- helpers --------------------------------------------------------------
    def _prizes(self):
        """Prizes REMAINING for (us, opponent). 0 remaining == that player won."""
        cur = self.obs_dict["current"]
        return len(cur["players"][0]["prize"]), len(cur["players"][1]["prize"])

    def _is_terminal(self):
        return (self.obs_dict is None) or (self.obs_dict["current"]["result"] >= 0)

    def _advance_until_our_turn(self):
        """Play opponent moves and auto-answer our non-decision prompts until
        either it's our real decision, or the game ends.
        Returns True if we now face a decision, False if the game ended."""
        steps = 0
        while self.obs_dict is not None and self.obs_dict["current"]["result"] < 0:
            steps += 1
            if steps > self.max_advance:
                # safety valve against a stuck engine loop
                return False
            yidx = self.obs_dict["current"]["yourIndex"]
            if yidx == 0:
                if self.obs_dict.get("select") is None:
                    action = self.my_deck        # setup / deck-return: not a policy choice
                else:
                    return True                  # <-- our real decision point
            else:
                action = self.opp_agent(self.obs_dict)
            self.obs_dict.pop("search_begin_input", None)
            self.obs_dict = battle_select(action)
        return False

    def _terminal_reward(self):
        if self.obs_dict is None:
            return 0.0                            # undecided/crashed game -> neutral
        result = self.obs_dict["current"]["result"]
        my_idx = 0  # we are always player 0
        if result == my_idx:
            return 1.0                            # we won
        elif result >= 0:
            return -1.0                           # opponent won
        return 0.0                               # shouldn't reach here if truly terminal

    def _pack(self, reward):
        if self.done or self._is_terminal() or self.obs_dict["current"]["yourIndex"] != 0 \
           or self.obs_dict.get("select") is None:
            return {"obs_dict": self.obs_dict, "options": [], "done": True, "reward": reward}
        obs = to_observation_class(self.obs_dict)
        return {"obs_dict": self.obs_dict, "obs": obs,
                "options": obs.select.option, "my_idx": obs.current.yourIndex,
                "done": False, "reward": reward}

    # -- gym-style API --------------------------------------------------------
    def reset(self):
        self.obs_dict, _ = battle_start(self.my_deck, self.opp_deck)
        self.done = False
        self._prev_prizes = self._prizes()
        if not self._advance_until_our_turn():
            self.done = True
        return self._pack(reward=0.0)

    def step(self, action_list):
        assert not self.done, "step() called on a finished episode; call reset()"
        self.obs_dict.pop("search_begin_input", None)
        self.obs_dict = battle_select(action_list)
        has_decision = self._advance_until_our_turn()

        reward = 0.0
        if self.obs_dict is not None:
            my_now, opp_now = self._prizes()
            my_prev, opp_prev = self._prev_prizes
            if self.reward_shaping:
                reward += (opp_prev - opp_now) * self.shape_scale   # we took prizes
                reward -= (my_prev - my_now) * self.shape_scale     # we lost prizes
            self._prev_prizes = (my_now, opp_now)

        if self._is_terminal() or not has_decision:
            self.done = True
            reward += self._terminal_reward()
        return self._pack(reward)

    def close(self):
        try:
            battle_finish()
        except Exception:
            pass

### 2. Actor-Critic network
The **scorer** mirrors your BC net (same 3 Linear layers, minus the Sigmoid) so BC weights load straight in; a softmax over its per-option logits is the policy. The **value head** estimates V(s) as the PPO baseline.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 2 — Actor-Critic network (policy + value), warm-started from BC
# ---------------------------------------------------------------------------
class ActorCritic(nn.Module):
    """Two heads sharing nothing but the same inputs:

    * scorer:  concat(state, option) -> ONE logit per option.
               softmax over the option set = the policy (a distribution over
               *which legal option to pick*). This mirrors your BC net exactly
               (same 3 Linear layers) but drops the Sigmoid, so BC weights load
               straight in.
    * value:   state features -> V(s), a scalar estimate of "how good is this
               position", used as the critic baseline for PPO.
    """

    def __init__(self, input_dim=None, state_dim=None):
        super().__init__()
        input_dim = input_dim if input_dim is not None else INPUT_DIM
        state_dim = state_dim if state_dim is not None else STATE_DIM
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.value_head = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, 1),
        )

    def option_logits(self, x):
        """x: [num_options, input_dim] -> logits: [num_options]"""
        return self.scorer(x).squeeze(-1)

    def value(self, s):
        """s: [state_dim] or [B, state_dim] -> V: scalar or [B]"""
        return self.value_head(s).squeeze(-1)


def warm_start_from_bc(actor_critic, bc_model):
    """Copy the imitation model's Linear weights into the actor's scorer so RL
    fine-tunes your BC policy instead of starting from random noise."""
    bc_layers = [m for m in bc_model.net if isinstance(m, nn.Linear)]
    ac_layers = [m for m in actor_critic.scorer if isinstance(m, nn.Linear)]
    with torch.no_grad():
        for src, dst in zip(bc_layers, ac_layers):
            dst.weight.copy_(src.weight)
            dst.bias.copy_(src.bias)
    return actor_critic

### 3. Timestep → inputs, and action sampling
Reuses your existing featurizers. `build_action_list` handles multi-pick selects (`minCount/maxCount`) by sampling one primary option and greedily filling the rest — a documented approximation.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 3 — Turning a timestep into policy inputs + sampling an action
# ---------------------------------------------------------------------------
def build_policy_inputs(ts):
    """From a (non-done) timestep, build:
        X : [num_options, INPUT_DIM]  concat(state, option) rows
        s : [STATE_DIM]               state-only features for the value head
    Reuses your EXISTING featurize_state / featurize_option unchanged."""
    obs_dict, obs = ts["obs_dict"], ts["obs"]
    options, my_idx = ts["options"], ts["my_idx"]
    sv = featurize_state(obs_dict)
    ov = np.array([featurize_option(obs, o, my_idx) for o in options], dtype=np.float32)
    X = np.concatenate([np.tile(sv, (len(options), 1)), ov], axis=1)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(sv, dtype=torch.float32)


def build_action_list(ts, logits, primary):
    """The engine sometimes wants k>1 picks (minCount/maxCount). The policy
    samples ONE primary option; if more are required we fill the rest greedily
    by score. (Approximation: log-prob covers only the primary pick — fine for
    a first RL pass; revisit if multi-select decisions dominate.)"""
    sel = ts["obs_dict"].get("select", {}) or {}
    min_c = safe_int(sel.get("minCount", 1))
    max_c = safe_int(sel.get("maxCount", 1))
    n = len(ts["options"])
    k = min(max(min_c, 1 if max_c >= 1 else 0), max_c, n)
    if k <= 1:
        return [primary]
    order = torch.argsort(logits, descending=True).tolist()
    picks = [primary] + [i for i in order if i != primary]
    return sorted(picks[:k])

### 4. Rollout
Plays one full game, storing everything PPO needs per step: option-feature matrix, state features, sampled action, log-prob, value, reward, done.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 4 — Rollout: play one full episode, storing transitions
# ---------------------------------------------------------------------------
def run_episode(actor_critic, env, deterministic=False):
    """Play one game. Returns (transitions, episode_reward).
    Each transition stores everything PPO needs to recompute gradients later:
    the option-feature matrix X, state features s, the sampled action index,
    its log-prob, the critic value, the reward, and the done flag."""
    ts = env.reset()
    transitions = []
    ep_reward = 0.0
    while not ts["done"]:
        X, s = build_policy_inputs(ts)
        with torch.no_grad():
            logits = actor_critic.option_logits(X)
            value = float(actor_critic.value(s))
        dist = torch.distributions.Categorical(logits=logits)
        primary = int(torch.argmax(logits)) if deterministic else int(dist.sample())
        logp = float(dist.log_prob(torch.tensor(primary)))

        action_list = build_action_list(ts, logits, primary)
        next_ts = env.step(action_list)

        transitions.append({
            "X": X, "s": s, "action": primary,
            "logp": logp, "value": value,
            "reward": next_ts["reward"], "done": next_ts["done"],
        })
        ep_reward += next_ts["reward"]
        ts = next_ts
    env.close()
    return transitions, ep_reward

### 5. Advantages (GAE) + PPO update
`compute_gae` fills `adv`/`ret` in place (handles episode boundaries via `done`). `ppo_update` does one clipped-objective update with value + entropy terms.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 5 — Advantages (GAE) + a compact PPO update
# ---------------------------------------------------------------------------
def compute_gae(transitions, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation. Adds 'adv' (advantage) and 'ret'
    (value target) to each transition in place. Handles episode boundaries via
    the done flag, so you can concatenate several episodes into one list."""
    adv = 0.0
    for t in reversed(range(len(transitions))):
        cur = transitions[t]
        next_v = 0.0 if cur["done"] else transitions[t + 1]["value"]
        delta = cur["reward"] + gamma * next_v - cur["value"]
        adv = delta + gamma * lam * (0.0 if cur["done"] else adv)
        cur["adv"] = adv
        cur["ret"] = adv + cur["value"]
    return transitions


def ppo_update(actor_critic, optimizer, transitions,
               epochs=4, clip=0.2, vf_coef=0.5, ent_coef=0.01):
    """One PPO update over the collected batch. Loops over transitions to
    recompute log-probs/values (clear but not the fastest — batching with
    padded option sets is the obvious optimization later)."""
    advs = torch.tensor([t["adv"] for t in transitions], dtype=torch.float32)
    advs = (advs - advs.mean()) / (advs.std() + 1e-8)
    rets = torch.tensor([t["ret"] for t in transitions], dtype=torch.float32)
    old_logps = torch.tensor([t["logp"] for t in transitions], dtype=torch.float32)

    stats = {}
    for _ in range(epochs):
        new_logps, entropies, values = [], [], []
        for t in transitions:
            logits = actor_critic.option_logits(t["X"])
            dist = torch.distributions.Categorical(logits=logits)
            new_logps.append(dist.log_prob(torch.tensor(t["action"])))
            entropies.append(dist.entropy())
            values.append(actor_critic.value(t["s"]))
        new_logps = torch.stack(new_logps)
        entropies = torch.stack(entropies)
        values = torch.stack(values)

        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advs
        surr2 = torch.clamp(ratio, 1 - clip, 1 + clip) * advs
        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = ((values - rets) ** 2).mean()
        entropy = entropies.mean()
        loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        stats = {"policy_loss": policy_loss.item(),
                 "value_loss": value_loss.item(),
                 "entropy": entropy.item()}
    return stats

### 6. Minimal PPO training loop
Each iteration: play a few games vs a randomly chosen opponent from the pool, compute advantages, run one PPO update. `random` opponent is skipped by default.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 6 — Putting it together: a minimal PPO training loop
# ---------------------------------------------------------------------------
def train_rl(actor_critic, opponents=None, iterations=20,
             episodes_per_iter=8, lr=3e-4, skip_random=True):
    """Minimal on-policy PPO loop. Each iteration: play a few games against a
    randomly chosen opponent, compute advantages, run one PPO update."""
    opponents = opponents if opponents is not None else OPPONENTS
    optimizer = optim.Adam(actor_critic.parameters(), lr=lr)
    pool = [(n, a, d) for n, (a, d) in opponents.items()
            if not (skip_random and n == "random")]

    history = []
    for it in range(iterations):
        batch, ep_rewards = [], []
        for _ in range(episodes_per_iter):
            name, opp_agent, opp_deck = random.choice(pool)
            env = PokemonRLEnv(opp_agent, opp_deck)
            tr, ep_r = run_episode(actor_critic, env)
            batch.extend(tr)
            ep_rewards.append(ep_r)
        compute_gae(batch)
        stats = ppo_update(actor_critic, optimizer, batch)
        mean_r = float(np.mean(ep_rewards))
        history.append(mean_r)
        print(f"iter {it:02d}  mean_ep_reward={mean_r:+.3f}  "
              f"pi_loss={stats['policy_loss']:+.3f}  "
              f"v_loss={stats['value_loss']:.3f}  ent={stats['entropy']:.3f}")
    return actor_critic, history

### 7. Smoke test (run this FIRST)
Exercises env + network + one PPO step on a single game. If this prints without error, the wiring is good and you can call `train_rl(...)`.


In [ ]:
# ---------------------------------------------------------------------------
# SECTION 7 — SMOKE TEST (run this FIRST, before any training)
# ---------------------------------------------------------------------------
def rl_smoke_test(bc_model=None):
    """Confirms the env + network wiring works end to end on a single game."""
    name, (opp_agent, opp_deck) = next(iter(OPPONENTS.items()))
    env = PokemonRLEnv(opp_agent, opp_deck)
    ts = env.reset()
    print(f"[smoke] opponent={name}  first-decision options={len(ts['options'])}  done={ts['done']}")

    ac = ActorCritic()
    if bc_model is not None:
        warm_start_from_bc(ac, bc_model)
        print("[smoke] warm-started actor from BC weights")

    tr, ep_r = run_episode(ac, env)
    print(f"[smoke] episode finished: steps={len(tr)}  episode_reward={ep_r:+.3f}")
    compute_gae(tr)
    opt = optim.Adam(ac.parameters(), lr=3e-4)
    stats = ppo_update(ac, opt, tr, epochs=1)
    print(f"[smoke] one PPO step OK: {stats}")
    return True

### How to run
```python
# 1) make sure your BC model exists (either the `model` var from training above,
#    or load it):
# bc = ImitationNet(INPUT_DIM); bc.load_state_dict(torch.load("model.pt")); bc.eval()

# 2) confirm the wiring:
rl_smoke_test(bc_model=model)          # uses the in-memory BC model

# 3) train:
agent = warm_start_from_bc(ActorCritic(), model)
agent, history = train_rl(agent, iterations=50, episodes_per_iter=8)
torch.save(agent.state_dict(), "model_rl.pt")
```
Start with a few iterations to sanity-check that `mean_ep_reward` is finite and
moving, then scale up. To ship it, load `model_rl.pt` in `main.py` and use
`ActorCritic.option_logits` for greedy option selection.


## Run RL Training

Execute these cells to actually train. Start with the smoke test, then scale up.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Smoke test — confirm env + network + PPO wiring
# ═══════════════════════════════════════════════════════════════════════════
rl_smoke_test(bc_model=bc_model)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Create RL agent, warm-started from BC
# ═══════════════════════════════════════════════════════════════════════════
agent = warm_start_from_bc(ActorCritic(), bc_model)
print("Agent created and warm-started from BC weights")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Train! Start with a few iterations to sanity-check, then scale up.
# ═══════════════════════════════════════════════════════════════════════════
agent, history = train_rl(agent, iterations=50, episodes_per_iter=8)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Save the trained RL model
# ═══════════════════════════════════════════════════════════════════════════
torch.save(agent.state_dict(), "model_rl.pt")
print("✓ RL model saved to model_rl.pt")
print(f"  Training history (last 5): {[round(h,3) for h in history[-5:]]}")